<div class="align-center">

<a href="https://rocm.docs.amd.com/projects/ai-developer-hub/en/latest/index.html"><img src="https://raw.githubusercontent.com/ROCm/gpuaidev/main/docs/images/rocm_logo.png" alt="ROCm AI Developer Hub" width="150" style="display:inline-block; margin-right: 20px;"></a>
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115" style="display:inline-block;"></a>

</div>

---

# Pokémon LLM Agent with Unsloth on AMD ROCm

**Authors:** Yue Yuan ([yueyuan](https://github.com/yueyuan), yueyuan@amd.com), Bill He ([billishyahao](https://github.com/billishyahao), bill.he@amd.com)

**Reviewers:** Eda Zhou, Mahdi Ghodsi ([Mahdi-CV](https://github.com/Mahdi-CV)), Tanina Obasi


**Knowledge level:** Intermediate

Fine-tune **Qwen3-4B** with **Unsloth** and **TRL** so the model predicts the next **Pokémon Showdown** action (`move …` or `switch …`) from replay logs. Training and inference target **AMD Instinct™ MI300 / MI355X** with **bfloat16**.

**Training data:** [`milkkarten/pokemon-showdown-replays-merged`](https://huggingface.co/datasets/milkkarten/pokemon-showdown-replays-merged)

**Pre-trained agents on Hugging Face:** [`pokemon-showdown-agent-tutorial-sft`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-sft) · [`pokemon-showdown-agent-tutorial-grpo`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-grpo) · [`pokemon-showdown-agent-full-sft`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-full-sft) · [`pokemon-showdown-agent-battle-grpo`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-battle-grpo)

Protocol parsing, GRPO rewards, and battle evaluation helpers are defined in **Step 3** and **Step 13b**.

This tutorial includes the following sections:

- [Prerequisites](#prerequisites)
- [Prepare the training environment](#prepare-the-training-environment)
- [Step 0 — Set paths and tutorial knobs](#step0)
- [Steps 1–4 — Install deps, ROCm check, Showdown protocol](#step1)
- [Steps 5–11 — Model pick, data, SFT smoke, mini-eval](#step5)
- [Step 12 — GRPO smoke (optional)](#step12)
- [Steps 14–15 — Battle eval and friendly play](#step14)



## Prerequisites

This tutorial was developed and tested using the following setup.

### Operating system

* **Ubuntu 22.04 / 24.04** (or compatible Linux with ROCm + Docker).

### Hardware

* **AMD Instinct™ GPU** with **≥ 24 GB** VRAM (Qwen3-4B + LoRA in bfloat16). See the [ROCm system requirements](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/reference/system-requirements.html).

### Software

* **ROCm** — install and verify:

  ```bash
  amd-smi
  ```

  > For ROCm 6.4 and earlier, use `rocm-smi` instead.

* **Docker** with GPU access. Configure non-root access if needed:

  ```bash
  sudo usermod -aG docker $USER
  newgrp docker
  docker run hello-world
  ```



* **Hugging Face** account (optional, for Hub models and dataset streaming): `hf auth login`

> **Important:** Use the official **`rocm/pytorch`** image below, then run Step 1 to install Unsloth and battle deps with pinned versions.


## Prepare the training environment

Follow these steps **before** running notebook cells.

### 1. Pull the official ROCm PyTorch image

```bash
docker pull rocm/pytorch:rocm7.1.1_ubuntu24.04_py3.12_pytorch_release_2.9.1
```

| Image | ROCm | PyTorch | Notes |
|-------|------|---------|-------|
| `rocm/pytorch:rocm7.1.1_ubuntu24.04_py3.12_pytorch_release_2.9.1` | 7.1.1 | 2.9.1 | **Recommended — tested on MI300X** |
| `rocm/pytorch:rocm7.2.4_ubuntu24.04_py3.12_pytorch_release_2.10.0` | 7.2.4 | 2.10.0 | Also validated on MI300X |
| Personal / unofficial images | — | — | **Not allowed** for Hub tutorials |

Unsloth, poke-env, JupyterLab, and Node.js are installed in **Step 1** (pinned versions).

### 2. Launch the container

Replace `/path/to/work` with a **host-writable** directory. Checkpoints and logs write under `/workspace`.

**On the host (once):**

```bash
mkdir -p /path/to/work
chmod 1777 /path/to/work    # rocm/pytorch runs as root by default
RENDER_GID=$(getent group render | cut -d: -f3)
VIDEO_GID=$(getent group video | cut -d: -f3)
```

Use the **host numeric GIDs** for `render` and `video` so `/dev/dri` is accessible (Step 2 GPU check). Publish **8888** (Jupyter) and **8000** (Showdown):

```bash
docker run -it --rm \
  --network=host \
  --device=/dev/kfd --device=/dev/dri \
  --group-add video \
  --group-add "${VIDEO_GID}" \
  --group-add "${RENDER_GID}" \
  --cap-add=SYS_PTRACE \
  --security-opt seccomp=unconfined \
  --ipc=host --shm-size=32G \
  -p 8888:8888 -p 8000:8000 \
  -v /path/to/work:/workspace \
  -w /workspace \
  rocm/pytorch:rocm7.1.1_ubuntu24.04_py3.12_pytorch_release_2.9.1 \
  bash
```

Download **this notebook** from the [AI Developer Hub GitHub repository](https://github.com/ROCm/gpuaidev) and place it in `/workspace`.

### 3. Quick sanity checks (optional)

```bash
python3 -c "import torch; print('cuda', torch.cuda.is_available(), torch.version.hip)"
touch /workspace/.write_probe && rm /workspace/.write_probe && echo 'workspace writable'
```

> If GPU is `False` or workspace probe fails, fix **render GID** / **directory permissions** before Step 2.

First Showdown start (Step 14) runs `git clone` + `npm install` under `/tmp/pokemon-showdown` — allow a few minutes on first run.

### 4. Launch Jupyter

Run **Step 1** first (installs Unsloth + deps), then:

```bash
jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --allow-root
```

Open the URL printed in the terminal (include the `?token=…` suffix).


<a id="step0"></a>

## Step 0 — Set paths and tutorial knobs

Run the **next two code cells first**. They set `WORK_ROOT`, ROCm-friendly Hugging Face cache paths, and optional environment knobs (`POKEMON_GPU`, tutorial mode, etc.).

Set `POKEMON_GPU` (or `CUDA_VISIBLE_DEVICES`) before running if GPU 0 is busy.


In [ ]:
# Optional tutorial knobs — edit before running, or set in the shell.
import os
os.environ.setdefault("POKEMON_GPU", "0")
os.environ.setdefault("POKEMON_TUTORIAL_MODE", "quick")  # quick | full
os.environ.setdefault("POKEMON_AGENT_TIER", "enthusiast")  # casual|enthusiast|competitive|champion (casual = tutorial SFT)
os.environ.setdefault("POKEMON_N_BATTLES", "10")
os.environ.setdefault("POKEMON_USE_HF_ASSETS", "1")
os.environ.setdefault("POKEMON_USE_POKE_ENV", "0")  # keep 0 on servers; Step 14 uses local Showdown
os.environ.setdefault("POKEMON_HF_BATTLE_GRPO", "GoldenGrapeGentleman1/pokemon-showdown-agent-battle-grpo")


In [ ]:
# Step 0 — Environment: ROCm/HIP-friendly defaults and writable Hugging Face cache.
import os
import platform
import shutil
from pathlib import Path

def pick_writable_dir(candidates):
    for candidate in candidates:
        path = Path(candidate)
        try:
            path.mkdir(parents=True, exist_ok=True)
            probe = path / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return path
        except Exception:
            continue
    raise RuntimeError(f"No writable directory found in: {candidates}")

cache_root = pick_writable_dir(["/shared-docker/.cache/huggingface", "/data/huggingface", "/tmp/pokemon-hf-cache"])
tmp_root = pick_writable_dir(["/shared-docker/.cache/tmp", "/data/tmp", "/tmp/pokemon-tmp"])

os.environ.setdefault("CUDA_VISIBLE_DEVICES", os.environ.get("POKEMON_GPU", "0"))
os.environ.pop("ROCR_VISIBLE_DEVICES", None)
os.environ.setdefault("TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL", "1")
os.environ.setdefault("PYTORCH_HIP_ALLOC_CONF", "expandable_segments:False")
os.environ.setdefault("UNSLOTH_SKIP_TORCHVISION_CHECK", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

os.environ["HF_HOME"] = str(cache_root)
os.environ["HF_DATASETS_CACHE"] = str(cache_root / "datasets")
os.environ["TMPDIR"] = str(tmp_root)
WORK_ROOT = Path.cwd().resolve()
os.environ["POKEMON_WORK_ROOT"] = str(WORK_ROOT)

Path(os.environ["HF_DATASETS_CACHE"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["TMPDIR"]).mkdir(parents=True, exist_ok=True)

cache_usage = shutil.disk_usage(cache_root)
print(f"platform={platform.platform()}")
print(f"HF_HOME={os.environ['HF_HOME']}")
print(f"WORK_ROOT={WORK_ROOT}")
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")
print(f"free_space_gb={cache_usage.free / (1024 ** 3):.1f}")


<a id="step1"></a>

## Step 1 — Install Unsloth and battle dependencies

The official **`rocm/pytorch`** image ships ROCm PyTorch only. This cell installs **Unsloth** and tutorial deps with **pinned versions** (re-run safely — skips packages already present).

Also installs **Node.js 22** (required by upstream Showdown) via NodeSource when missing.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys
import shutil
from pathlib import Path

PIN = {
    "unsloth": "2026.1.4",
    "poke-env": "0.15.0",
    "jupyterlab": "4.4.1",
    "ipywidgets": "8.1.7",
    "datasets": "3.6.0",
}
BNB_WHEEL = (
    "https://github.com/bitsandbytes-foundation/bitsandbytes/releases/download/"
    "continuous-release_main/bitsandbytes-1.33.7.preview-py3-none-manylinux_2_24_x86_64.whl"
)
IMPORT_NAMES = {"poke-env": "poke_env", "jupyterlab": "jupyterlab", "ipywidgets": "ipywidgets"}

def _pkg_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

def _importable(pkg: str) -> bool:
    mod = IMPORT_NAMES.get(pkg, pkg.replace("-", "_"))
    return importlib.util.find_spec(mod) is not None

def pip_install(args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args, check=True)

import torch
print(f"torch={torch.__version__} hip={getattr(torch.version, 'hip', None)}")

if not _importable("unsloth"):
    # rocm/pytorch includes ROCm PyTorch; install the pinned Unsloth build.
    print(f"Installing unsloth=={PIN['unsloth']} ...")
    pip_install([f"unsloth=={PIN['unsloth']}", f"unsloth_zoo=={PIN['unsloth']}"])
    try:
        pip_install(["--force-reinstall", "--no-cache-dir", "--no-deps", BNB_WHEEL])
    except subprocess.CalledProcessError:
        print("  (bitsandbytes ROCm wheel optional — 16-bit training still works)")
else:
    print(f"unsloth already installed ({_pkg_version('unsloth')})")

# Keep unsloth_zoo aligned with unsloth; always regenerate compiled trainers.
_cache_dir = Path.cwd() / "unsloth_compiled_cache"
if _cache_dir.exists():
    print("Clearing unsloth_compiled_cache (regenerated on first trainer use) ...")
    shutil.rmtree(_cache_dir, ignore_errors=True)

for pkg, ver in PIN.items():
    if pkg == "unsloth":
        continue
    if not _importable(pkg):
        pip_install([f"{pkg}=={ver}"])

node = shutil.which("node")
need_node = True
if node:
    import re
    out = subprocess.run([node, "--version"], capture_output=True, text=True, check=False)
    need_node = int(re.match(r"v?(\d+)", out.stdout.strip() or "0").group(1)) < 22
if need_node:
    print("Installing Node.js 22 (Showdown server) ...")
    subprocess.run(
        ["bash", "-lc", "set -e; curl -fsSL https://deb.nodesource.com/setup_22.x | bash -; apt-get install -y -qq nodejs"],
        check=True,
    )

for pkg in ("unsloth", "poke-env", "jupyterlab", "ipywidgets", "transformers", "trl", "datasets"):
    print(f"  {pkg}={_pkg_version(pkg)}")
print("Step 1 OK")


## Step 2 — Validate the AMD ROCm runtime
**Tutorial content — hardware smoke test.** The next cell imports in **Unsloth-first** order, prints package versions, and checks that **PyTorch** reports a GPU (ROCm builds typically expose HIP via the CUDA-like API). If this fails, fix the container or driver stack before training.


In [ ]:
import unsloth
import datasets
import huggingface_hub
import torch
import transformers
import trl


datasets.config.HF_DATASETS_CACHE = os.environ["HF_DATASETS_CACHE"]

print(f"torch={torch.__version__}")
print(f"transformers={transformers.__version__}")
print(f"trl={trl.__version__}")
print(f"datasets={datasets.__version__}")
print(f"huggingface_hub={huggingface_hub.__version__}")
print(f"unsloth={getattr(unsloth, '__version__', 'unknown')}")
print(f"hip_version={getattr(torch.version, 'hip', None)}")

if not torch.cuda.is_available():
    raise RuntimeError("No ROCm-visible GPU found. This notebook expects an AMD GPU environment.")

print(f"gpu_count={torch.cuda.device_count()}")
print(f"gpu_name={torch.cuda.get_device_name(0)}")
print(f"bf16_supported={torch.cuda.is_bf16_supported()}")


## Step 3 — Showdown eval helpers (inline)

These cells define every helper used later — **all logic is inline in this notebook**. They:

1. Parse Showdown protocol lines (`|move|`, `|request|`, …)
2. Validate agent outputs (`move …` / `switch …`)
3. Build mini-eval metrics and GRPO reward functions
4. Start a **local** Showdown server for Step 14–15

| Helper | Purpose |
|--------|---------|
| `validate_action_against_log` | Format + optional `|request|` legality |
| `build_test_samples` / `eval_showdown_agent_batch` | Replay mini-eval |
| `make_showdown_grpo_reward` | GRPO training reward |
| `bootstrap_local_showdown` | Local server on port 8000 |


### 3a — Showdown protocol + GRPO eval helpers

Run the next cell to define all parsing, validation, mini-eval, and GRPO reward functions.


In [ ]:
# Showdown protocol and GRPO eval helpers
"""
Shared evaluation helpers for the Pokémon Showdown LLM agent tutorial.

Metrics per sample:
  - valid format: output parses as move/switch (extract_cmd)
  - type_match: predicted move vs switch matches ground truth category
  - exact_match: normalized move/switch target string matches ground truth
"""
from __future__ import annotations

import json
import os
import random
import re
import time
from pathlib import Path
from typing import Any, Callable, Iterator

SYSTEM_TEMPLATE = (
    "You are a Pokemon Showdown battle AI. You play as {side}. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)


def postprocess_agent_response(response: str) -> str:
    """Strip Qwen3 thinking wrapper and chat markers (matches notebook inference)."""
    response = response.replace("<|im_end|>", "").strip()
    response = re.sub(
        r"(?s)^<think>.*?</think>\s*", "", response
    ).strip()
    return response


def showdown_fields(line: str) -> list[str]:
    """
    Split a Pokemon Showdown protocol line into fields.

    Lines usually look like ``|player|p1|Name|...``. A plain ``split('|')`` leaves
    leading empty segments; we strip those so the message type is always ``fields[0]``.
    """
    parts = line.strip().split("|")
    while parts and parts[0] == "":
        parts = parts[1:]
    return parts


# --- Tutorial / single-turn legality ---


def parse_showdown_action_line(line: str) -> dict[str, Any] | None:
    """First line: 'move Name ...' / 'switch Name ...' with optional trailing 'terastallize'."""
    line = line.strip()
    if not line:
        return None
    m = re.match(r"^(move|switch)\s+(.+)$", line, flags=re.IGNORECASE)
    if not m:
        return None
    verb = m.group(1).lower()
    rest = m.group(2).strip()
    tera = False
    low = rest.lower()
    if low.endswith(" terastallize"):
        tera = True
        rest = rest[: -len(" terastallize")].strip()
    if not rest:
        return None
    return {"verb": verb, "target": rest, "tera": tera}


def extract_last_request_json(log_text: str) -> dict[str, Any] | None:
    """Parse the last |request|{...} line (Showdown turn JSON)."""
    for raw in reversed(log_text.strip().splitlines()):
        raw = raw.strip()
        if not raw.startswith("|request|"):
            continue
        rest = raw[len("|request|") :]
        if rest.startswith("{"):
            blob = rest
        else:
            idx = rest.find("{")
            if idx == -1:
                continue
            blob = rest[idx:]
        try:
            return json.loads(blob)
        except json.JSONDecodeError:
            continue
    return None


def species_from_request_pokemon(p: dict[str, Any]) -> str:
    det = (p.get("details") or "").strip()
    if det:
        return det.split(",")[0].strip()
    ident = (p.get("ident") or "").strip()
    if ":" in ident:
        return ident.split(":", 1)[1].strip()
    return ""


def legal_moves_from_request(req: dict[str, Any]) -> list[str]:
    names: list[str] = []
    if not req or "active" not in req:
        return names
    for slot in req.get("active") or []:
        for m in slot.get("moves") or []:
            if m.get("disabled"):
                continue
            mv = m.get("move")
            if mv:
                names.append(mv.casefold())
    return names


def legal_switch_species_from_request(req: dict[str, Any]) -> list[str]:
    out: list[str] = []
    side = (req or {}).get("side") or {}
    for p in side.get("pokemon") or []:
        cond = (p.get("condition") or "").lower()
        if "fnt" in cond or cond.startswith("0 "):
            continue
        if p.get("active"):
            continue
        sp = species_from_request_pokemon(p)
        if sp:
            out.append(sp.casefold())
    return out


def can_terastallize_from_request(req: dict[str, Any]) -> bool:
    if not req or "active" not in req:
        return False
    for slot in req.get("active") or []:
        if slot.get("canTerastallize"):
            return True
    return False


def p2_roster_and_active_from_log(log_text: str) -> tuple[list[str], str | None]:
    """Species names from |switch|p2*| lines; active p2a from the last such line."""
    roster: list[str] = []
    active_p2a: str | None = None
    for raw in log_text.splitlines():
        f = showdown_fields(raw)
        if len(f) < 3 or f[0] != "switch":
            continue
        slot = f[1]
        if not re.match(r"p2[a-z]?:", slot):
            continue
        species = f[2].split(",")[0].strip()
        if not species:
            continue
        roster.append(species)
        if slot.startswith("p2a:"):
            active_p2a = species
    return roster, active_p2a


def validate_action_against_log(action_line: str, log_text: str) -> dict[str, Any]:
    """
    Structure + optional |request| legality. Keys are stable for notebook printing.
    """
    roster, active = p2_roster_and_active_from_log(log_text)
    roster_set = {s.casefold() for s in roster}
    req = extract_last_request_json(log_text)
    parsed = parse_showdown_action_line(action_line)
    if not parsed:
        return {
            "parsed": None,
            "structure_ok": False,
            "single_line_ok": False,
            "request_present": req is not None,
            "move_legal_in_request": None,
            "switch_legal_in_request": None,
            "tera_legal_in_request": None,
            "switch_target_ok": None,
            "move_name_nonempty": None,
            "notes": "does not match move/switch line grammar",
        }
    lines = [ln for ln in action_line.splitlines() if ln.strip()]
    single = len(lines) <= 1
    mv_ok = None
    sw_ok = None
    move_legal = None
    switch_legal = None
    tera_legal = None
    if parsed["verb"] == "move":
        mv_ok = bool(parsed["target"].strip())
        if req:
            legal = set(legal_moves_from_request(req))
            move_legal = parsed["target"].casefold() in legal
            if parsed["tera"]:
                tera_legal = can_terastallize_from_request(req)
            else:
                tera_legal = True
    if parsed["verb"] == "switch":
        tgt = parsed["target"].casefold()
        sw_ok = tgt in roster_set and (active is None or tgt != active.casefold())
        if req:
            legal_sw = set(legal_switch_species_from_request(req))
            switch_legal = tgt in legal_sw
    return {
        "parsed": parsed,
        "structure_ok": True,
        "single_line_ok": single,
        "request_present": req is not None,
        "move_legal_in_request": move_legal,
        "switch_legal_in_request": switch_legal,
        "tera_legal_in_request": tera_legal,
        "switch_target_ok": sw_ok,
        "move_name_nonempty": mv_ok,
        "notes": "ok",
    }


def tutorial_demo_log_with_request() -> str:
    """
    Fixed battle prefix + synthetic |request| so tutorials can check move legality.
    Corviknight four moves are OU-plausible; Earthquake is intentionally NOT listed.
    """
    req = {
        "active": [
            {
                "moves": [
                    {"move": "Brave Bird", "id": "bravebird", "pp": 24, "maxpp": 24, "disabled": False},
                    {"move": "Iron Head", "id": "ironhead", "pp": 24, "maxpp": 24, "disabled": False},
                    {"move": "Roost", "id": "roost", "pp": 8, "maxpp": 8, "disabled": False},
                    {"move": "U-turn", "id": "uturn", "pp": 24, "maxpp": 24, "disabled": False},
                ],
                "canTerastallize": "Flying",
            }
        ],
        "side": {
            "pokemon": [
                {"ident": "p2: Corviknight", "details": "Corviknight, M, L50", "condition": "100/100", "active": True},
                {"ident": "p2: Dragapult", "details": "Dragapult, M, L50", "condition": "100/100", "active": False},
            ]
        },
    }
    return (
        "|player|p1|Player1|266|1500\n"
        "|player|p2|Player2|1|1500\n"
        "|teamsize|p1|6\n"
        "|teamsize|p2|6\n"
        "|gen|9\n"
        "|tier|[Gen 9] OU\n"
        "|\n"
        "|start\n"
        "|switch|p1a: Garchomp|Garchomp, M|100/100\n"
        "|switch|p2a: Corviknight|Corviknight, M|100/100\n"
        "|turn|1\n"
        "|move|p1a: Garchomp|Earthquake|p2a: Corviknight\n"
        "|-immune|p2a: Corviknight\n"
        "|request|"
        + json.dumps(req, separators=(",", ":"))
        + "\n|turn|2"
    )


def extract_winner_side(log_text: str):
    winner = None
    players = {}
    for line in log_text.split("\n"):
        f = showdown_fields(line)
        if len(f) >= 3 and f[0] == "player":
            players[f[1]] = f[2]
        if len(f) >= 2 and f[0] == "win":
            winner = f[1]

    if not winner:
        return None, None

    for side, name in players.items():
        if name == winner:
            return side, winner
    return None, winner


# --- GRPO tutorial dataset (Showdown |request| prompts) ---

HF_GRPO_DATASET_DEFAULT = "GoldenGrapeGentleman1/pokemon-showdown-grpo-tutorial"


def extract_grpo_log_slice(full_log: str, max_chars: int = 4500) -> str | None:
    """Keep log tail ending at the last |request| line (required for legality reward)."""
    lines = full_log.strip().splitlines()
    req_indices = [i for i, ln in enumerate(lines) if ln.strip().startswith("|request|")]
    if not req_indices:
        return None
    req_i = req_indices[-1]
    start = max(0, req_i - 50)
    chunk = lines[start : req_i + 1]
    if req_i + 1 < len(lines):
        f = showdown_fields(lines[req_i + 1])
        if f and f[0] == "turn":
            chunk.append(lines[req_i + 1])
    log = "\n".join(chunk)
    if len(log) > max_chars:
        log = log[-max_chars:]
    return log


def grpo_record_from_replay_row(row: dict[str, Any]) -> dict[str, Any] | None:
    """One GRPO training record from a streaming replay row."""
    side, _ = extract_winner_side(row["log"])
    if not side:
        return None
    log_slice = extract_grpo_log_slice(row["log"])
    if not log_slice or "|request|" not in log_slice:
        return None
    return {
        "side": side,
        "log": log_slice,
        "rating": row.get("rating"),
        "format": row.get("formatid"),
    }


def grpo_chat_prompt(
    record: dict[str, Any],
    tokenizer,
    *,
    add_generation_prompt: bool = True,
    concise: bool = True,
) -> str:
    system = SYSTEM_TEMPLATE.format(side=record["side"])
    if concise and record.get("schema", GRPO_SCHEMA_SFT) == GRPO_SCHEMA_SFT:
        system += (
            " Reply with exactly one action line (move or switch). "
            "No analysis or explanation."
        )
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": record["log"]},
    ]
    return tokenizer.apply_chat_template(
        msgs,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=False,
    )


def collect_grpo_records_from_stream(
    dataset: Iterator[dict[str, Any]],
    num_records: int,
    *,
    min_rating: int = 1400,
    require_gen9: bool = True,
    max_scans: int = 120_000,
) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    scanned = 0
    for row in dataset:
        scanned += 1
        if len(records) >= num_records:
            break
        if scanned > max_scans:
            break
        fmt = (row.get("formatid") or "").lower().replace(" ", "")
        if require_gen9 and "gen9" not in fmt:
            continue
        if (row.get("rating") or 0) < min_rating:
            continue
        rec = grpo_record_from_replay_row(row)
        if rec:
            records.append(rec)
    return records


def save_grpo_jsonl(records: list[dict[str, Any]], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def load_grpo_jsonl(path: str) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def load_grpo_tutorial_records(
    *,
    hf_dataset: str = HF_GRPO_DATASET_DEFAULT,
    split: str = "demo",
    local_path: str | None = None,
    use_hf: bool = True,
) -> list[dict[str, Any]]:
    """Load pre-built GRPO records: local jsonl first, then Hugging Face Dataset."""
    if local_path and os.path.isfile(local_path):
        return load_grpo_jsonl(local_path)
    if use_hf:
        from datasets import load_dataset

        try:
            ds = load_dataset(hf_dataset, split=split)
            return [dict(row) for row in ds]
        except Exception:
            ds = load_dataset(
                hf_dataset,
                data_files={split: f"data/{split}.jsonl"},
                split=split,
            )
            return [dict(row) for row in ds]
    raise FileNotFoundError(
        f"No GRPO data at {local_path!r} and use_hf=False. "
        "Run Step 12a (replay stream) or set POKEMON_USE_HF_ASSETS=1."
    )


def build_test_samples(
    dataset: Iterator[dict[str, Any]],
    num_samples: int,
    *,
    seed: int = 42,
    min_rating: int = 1400,
    require_gen9: bool = True,
    format_substr: str | None = None,
    max_scans: int = 400_000,
) -> list[dict[str, Any]]:
    """
    Build (prompt, gt_action, side, ...) tuples: winner's action at a random mid-game turn.
    """
    random.seed(seed)
    samples: list[dict[str, Any]] = []
    scanned = 0

    for row in dataset:
        scanned += 1
        if len(samples) >= num_samples:
            break
        if scanned > max_scans:
            break

        fmt = (row.get("formatid") or "").lower().replace(" ", "")
        if require_gen9 and "gen9" not in fmt:
            continue
        if format_substr and format_substr.lower().replace(" ", "") not in fmt:
            continue
        if (row.get("rating") or 0) < min_rating:
            continue

        log_text = row["log"]
        winner_side, _winner_name = extract_winner_side(log_text)
        if not winner_side:
            continue

        lines = log_text.strip().split("\n")

        turn_positions: list[tuple[int, int]] = []
        for i, line in enumerate(lines):
            f = showdown_fields(line)
            if len(f) >= 2 and f[0] == "turn":
                try:
                    turn_positions.append((int(f[1]), i))
                except ValueError:
                    pass

        if not turn_positions or len(turn_positions) <= 3:
            continue

        target_turn_idx = random.randint(2, len(turn_positions) - 2)
        _turn_num, turn_line_idx = turn_positions[target_turn_idx]

        if target_turn_idx + 1 < len(turn_positions):
            next_turn_line = turn_positions[target_turn_idx + 1][1]
        else:
            next_turn_line = len(lines)

        gt_action = None
        for j in range(turn_line_idx + 1, next_turn_line):
            f = showdown_fields(lines[j])
            if len(f) < 3:
                continue
            if f[0] == "move" and f[1].startswith(f"{winner_side}a:"):
                tera = ""
                start_look = max(0, j - 3)
                end_look = min(len(lines), j + 3)
                if any(
                    "terastallize" in lines[k] and winner_side in lines[k]
                    for k in range(start_look, end_look)
                ):
                    tera = " terastallize"
                gt_action = f"move {f[2]}{tera}"
                break
            if f[0] == "switch" and f[1].startswith(f"{winner_side}a:"):
                pokemon = f[1].split(": ", 1)[1] if ": " in f[1] else f[1]
                gt_action = f"switch {pokemon}"
                break

        if not gt_action:
            continue

        log_prefix = "\n".join(lines[: turn_line_idx + 1])
        samples.append(
            {
                "rating": row["rating"],
                "format": row["formatid"],
                "side": winner_side,
                "prompt": log_prefix,
                "gt_action": gt_action,
            }
        )

    return samples


# --- GRPO SFT-format replay dataset (no |request| required) ---

GRPO_SCHEMA_SFT = "sft_replay"
GRPO_SCHEMA_REQUEST = "request"


def load_type_chart(path: str | None = None) -> dict[str, Any]:
    """Minimal type chart for GRPO type-effectiveness bonus (no external file)."""
    if path:
        p = Path(path)
        if p.is_file():
            return json.loads(p.read_text(encoding="utf-8"))
    return {
        "super_effective": {
            "Flying": ["Grass", "Fighting", "Bug"],
            "Ground": ["Electric", "Fire", "Poison", "Rock", "Steel"],
            "Electric": ["Water", "Flying"],
            "Fire": ["Grass", "Ice", "Bug", "Steel"],
            "Water": ["Fire", "Ground", "Rock"],
            "Ice": ["Grass", "Ground", "Flying", "Dragon"],
            "Fighting": ["Normal", "Ice", "Rock", "Dark", "Steel"],
            "Dark": ["Psychic", "Ghost"],
            "Fairy": ["Fighting", "Dragon", "Dark"],
        }
    }


def move_type_hint(move_name: str) -> str | None:
    low = move_name.lower()
    hints = {
        "thunder": "Electric",
        "volt": "Electric",
        "bolt": "Electric",
        "earth": "Ground",
        "quake": "Ground",
        "flame": "Fire",
        "fire": "Fire",
        "heat": "Fire",
        "hydro": "Water",
        "surf": "Water",
        "aqua": "Water",
        "ice": "Ice",
        "freeze": "Ice",
        "brave bird": "Flying",
        "bird": "Flying",
        "iron head": "Steel",
        "knock off": "Dark",
        "close combat": "Fighting",
        "moonblast": "Fairy",
        "shadow ball": "Ghost",
        "psychic": "Psychic",
        "dragon": "Dragon",
        "u-turn": "Bug",
        "uturn": "Bug",
    }
    for key, typ in hints.items():
        if key in low:
            return typ
    return None


def defender_types_from_log(log_text: str, *, side: str = "p1") -> list[str]:
    """Heuristic opponent types from |switch| lines (opponent = p1 when we play p2)."""
    opp = "p1" if side == "p2" else "p2"
    types: list[str] = []
    for line in log_text.splitlines():
        if f"|switch|{opp}a:" not in line:
            continue
        low = line.lower()
        for typ in (
            "Water",
            "Fire",
            "Electric",
            "Ground",
            "Dragon",
            "Flying",
            "Steel",
            "Grass",
            "Ice",
            "Fairy",
            "Dark",
            "Ghost",
            "Fighting",
            "Psychic",
            "Bug",
            "Rock",
            "Poison",
            "Normal",
        ):
            if typ.lower() in low and typ not in types:
                types.append(typ)
    return types


def type_effectiveness_bonus(
    move_name: str,
    log_text: str,
    type_chart: dict[str, Any],
    *,
    side: str = "p2",
) -> float:
    move_type = move_type_hint(move_name)
    if not move_type or not type_chart.get("super_effective"):
        return 0.0
    defenders = defender_types_from_log(log_text, side=side)
    if not defenders:
        return 0.0
    se = type_chart.get("super_effective", {}).get(move_type, [])
    bonus = 0.0
    for dt in defenders:
        if dt in se:
            bonus += 1.0
        elif move_type == "Ground" and dt == "Electric":
            bonus += 1.0
    return min(bonus, 2.0)


def truncate_log_tail(log_text: str, max_lines: int = 35, max_chars: int = 3500) -> str:
    """Keep recent Showdown lines for GRPO (shorter prompts, less thinking)."""
    lines = log_text.strip().splitlines()
    if len(lines) > max_lines:
        lines = lines[-max_lines:]
    out = "\n".join(lines)
    if len(out) > max_chars:
        out = out[-max_chars:]
    return out


def grpo_record_from_test_sample(sample: dict[str, Any]) -> dict[str, Any]:
    return {
        "side": sample["side"],
        "log": truncate_log_tail(sample["prompt"]),
        "gt_action": sample["gt_action"],
        "rating": sample.get("rating"),
        "format": sample.get("format"),
        "schema": GRPO_SCHEMA_SFT,
        "source": "replay",
    }


def collect_grpo_sft_records_from_stream(
    dataset: Iterator[dict[str, Any]],
    num_records: int,
    *,
    seed: int = 3407,
    min_rating: int = 1200,
    require_gen9: bool = True,
    max_scans: int = 400_000,
    prefer_moves: bool = True,
) -> list[dict[str, Any]]:
    """
    Build diverse GRPO records from replay mid-game prefixes (same as SFT / mini-eval).
    Does not require |request| in logs.
    """
    oversample = max(num_records * 3, num_records + 32)
    samples = build_test_samples(
        dataset,
        oversample,
        seed=seed,
        min_rating=min_rating,
        require_gen9=require_gen9,
        max_scans=max_scans,
    )
    if prefer_moves:
        move_samples = [s for s in samples if s.get("gt_action", "").lower().startswith("move ")]
        if len(move_samples) >= max(4, num_records // 2):
            samples = move_samples + [s for s in samples if s not in move_samples]

    seen: set[str] = set()
    records: list[dict[str, Any]] = []
    for sample in samples:
        key = sample["prompt"][:400]
        if key in seen:
            continue
        seen.add(key)
        records.append(grpo_record_from_test_sample(sample))
        if len(records) >= num_records:
            break
    return records


def log_from_chat_prompt(prompt: str | list) -> str:
    if isinstance(prompt, list):
        for msg in prompt:
            if isinstance(msg, dict) and msg.get("role") == "user":
                return str(msg.get("content", ""))
        return str(prompt)
    marker = "<|im_start|>user\n"
    if marker in prompt:
        body = prompt.split(marker, 1)[1]
        return body.split("<|im_start|>", 1)[0].strip() if "<|im_start|>" in body else body.strip()
    return str(prompt)


def action_line_from_completion(raw: str) -> tuple[str, str]:
    text = postprocess_agent_response(raw)
    cmd, _ = extract_cmd(text)
    if cmd:
        return text, cmd
    line = text.strip().splitlines()[0].strip() if text.strip() else ""
    return text, line


def make_showdown_grpo_reward(type_chart: dict[str, Any] | None = None):
    """
    GRPO reward aligned with SFT eval: format + type effectiveness + gt_action shaping + concise output.
    Dataset rows should include optional ``gt_action`` (passed through by TRL GRPOTrainer).
    """
    chart = type_chart or {}

    def reward_func(prompts, completions, gt_action=None, side=None, **kwargs):
        rewards: list[float] = []
        n = len(prompts)
        gt_list: list[str | None]
        if gt_action is None:
            gt_list = [None] * n
        elif isinstance(gt_action, list):
            gt_list = list(gt_action)
        else:
            gt_list = [gt_action] * n
        side_list: list[str | None]
        if side is None:
            side_list = [None] * n
        elif isinstance(side, list):
            side_list = list(side)
        else:
            side_list = [side] * n

        for prompt, completion, gt, play_side in zip(prompts, completions, gt_list, side_list):
            raw = completion[0]["content"] if isinstance(completion, list) else str(completion)
            text, line = action_line_from_completion(raw)
            if not line:
                cmd, _ = extract_cmd(raw)
                if cmd:
                    line = cmd
                    text = cmd
            log = log_from_chat_prompt(prompt if isinstance(prompt, str) else str(prompt))
            side_val = play_side or "p2"
            r = 0.0

            if not line:
                low = raw.lower()
                if "move " in low or "switch " in low:
                    r += 0.5
                r -= 2.0
                rewards.append(round(r, 2))
                continue

            c = validate_action_against_log(line, log)
            if not c["structure_ok"]:
                rewards.append(-3.0)
                continue

            r += 2.0
            if len([ln for ln in text.splitlines() if ln.strip()]) <= 1:
                r += 0.5

            if line.lower().startswith("move "):
                r += type_effectiveness_bonus(
                    line.split(" ", 1)[1], log, chart, side=side_val
                )

            if gt:
                pred_cmd, pred_type = extract_cmd(line)
                gt_cmd, gt_type = extract_cmd(str(gt))
                if pred_cmd and gt_cmd:
                    p_name = " ".join(pred_cmd.lower().split()[1:])
                    g_name = " ".join(gt_cmd.lower().split()[1:])
                    if p_name == g_name:
                        r += 3.0
                    elif pred_type and gt_type and pred_type == gt_type:
                        r += 1.5

            raw_words = len(raw.split())
            if raw_words <= 12:
                r += 1.5
            elif raw_words <= 40:
                r += 0.5
            elif raw_words > 80:
                r -= 1.5

            rewards.append(round(r, 2))
        return rewards

    reward_func.__name__ = "showdown_reward_func"
    return reward_func


def grpo_rows_for_trainer(
    records: list[dict[str, Any]],
    tokenizer,
    *,
    max_prompt_tokens: int = 2048,
    prefer_max_tokens: int = 1536,
    as_messages: bool = False,
) -> list[dict[str, Any]]:
    """Convert jsonl records to GRPOTrainer rows (prompt + gt_action + side)."""
    seen: set[str] = set()
    candidates: list[tuple[int, dict[str, Any]]] = []
    for rec in records:
        log_key = rec.get("log", "")[:400]
        if log_key in seen:
            continue
        seen.add(log_key)
        system = SYSTEM_TEMPLATE.format(side=rec.get("side", "p2"))
        if rec.get("schema", GRPO_SCHEMA_SFT) == GRPO_SCHEMA_SFT:
            system += (
                " Reply with exactly one action line (move or switch). "
                "No analysis or explanation."
            )
        elif rec.get("schema") == "battle_outcome":
            system += " Pick one action from the legal list."
        if as_messages:
            prompt_field: str | list = [
                {"role": "system", "content": system},
                {"role": "user", "content": rec["log"]},
            ]
            rendered = tokenizer.apply_chat_template(
                prompt_field,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        else:
            rendered = grpo_chat_prompt(rec, tokenizer)
            prompt_field = rendered
        n_tok = len(tokenizer(rendered, add_special_tokens=False)["input_ids"])
        if n_tok > max_prompt_tokens or n_tok > prefer_max_tokens:
            continue
        row = {
            "prompt": prompt_field,
            "gt_action": rec.get("gt_action") or "",
            "side": rec.get("side") or "p2",
        }
        if rec.get("expert_action"):
            row["expert_action"] = rec["expert_action"]
        if "battle_won" in rec:
            row["battle_won"] = bool(rec["battle_won"])
        if "outcome_sparse" in rec:
            row["outcome_sparse"] = bool(rec["outcome_sparse"])
        candidates.append((n_tok, row))
    candidates.sort(key=lambda x: x[0])
    return [row for _, row in candidates]


def extract_cmd(text: str):
    """Extract 'move X' or 'switch X' from model output."""
    m = re.search(
        r"(move\s+[\w\-]+(?:\s+[\w\-]+)?|switch\s+[\w\-]+(?:\s+[\w\-]+)?)",
        text,
        re.IGNORECASE,
    )
    if m:
        cmd = m.group(1).strip()
        cmd_type = "move" if cmd.lower().startswith("move") else "switch"
        return cmd, cmd_type
    return None, None


def eval_showdown_agent_batch(
    model,
    tokenizer,
    samples: list[dict[str, Any]],
    *,
    max_new_tokens: int = 30,
    use_cache: bool = False,
    device: str = "cuda",
    progress: Callable[[int, int, dict[str, Any]], None] | None = None,
) -> tuple[dict[str, int], list[dict[str, Any]]]:
    """
    Run inference on prepared samples; return aggregate metrics and per-row details.
    Prompting uses `apply_chat_template` to match the training notebook.
    """
    import torch

    metrics = {
        "total": 0,
        "move_pred": 0,
        "switch_pred": 0,
        "invalid_pred": 0,
        "type_match": 0,
        "exact_match": 0,
    }
    rows: list[dict[str, Any]] = []

    for i, sample in enumerate(samples):
        messages = [
            {
                "role": "system",
                "content": SYSTEM_TEMPLATE.format(side=sample["side"]),
            },
            {"role": "user", "content": sample["prompt"]},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(device)

        t0 = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=use_cache,
            )

        full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
        if "<|im_start|>assistant\n" in full_response:
            response = full_response.split("<|im_start|>assistant\n")[-1]
        else:
            response = tokenizer.decode(
                outputs[0][inputs.input_ids.shape[-1] :], skip_special_tokens=True
            )
        response = postprocess_agent_response(response)
        elapsed = time.time() - t0

        pred_cmd, pred_type = extract_cmd(response)
        gt_cmd, gt_type = extract_cmd(sample["gt_action"])

        metrics["total"] += 1
        if pred_type == "move":
            metrics["move_pred"] += 1
        elif pred_type == "switch":
            metrics["switch_pred"] += 1
        else:
            metrics["invalid_pred"] += 1

        type_match = (pred_type == gt_type) if (pred_type and gt_type) else False
        exact_match = False
        if pred_cmd and gt_cmd:
            p_name = " ".join(pred_cmd.lower().split()[1:])
            g_name = " ".join(gt_cmd.lower().split()[1:])
            if p_name == g_name:
                exact_match = True

        if type_match:
            metrics["type_match"] += 1
        if exact_match:
            metrics["exact_match"] += 1

        row = {
            "i": i,
            "rating": sample["rating"],
            "format": sample["format"],
            "gt_action": sample["gt_action"],
            "response_head": response[:500],
            "pred_cmd": pred_cmd,
            "pred_type": pred_type,
            "type_match": type_match,
            "exact_match": exact_match,
            "elapsed_s": elapsed,
        }
        rows.append(row)
        if progress:
            progress(i + 1, len(samples), row)

    return metrics, rows


def print_metrics_summary(metrics: dict[str, int], title: str = "Evaluation summary") -> None:
    t = metrics["total"]
    if t == 0:
        print(f"{title}: no samples")
        return
    valid = metrics["move_pred"] + metrics["switch_pred"]
    print(f"{'=' * 60}")
    print(title)
    print(f"{'=' * 60}")
    print(f"Total samples: {t}")
    print(
        f"Valid-format predictions: {valid}/{t} ({100.0 * valid / t:.1f}%) "
        f"(move={metrics['move_pred']}, switch={metrics['switch_pred']}, invalid={metrics['invalid_pred']})"
    )
    print(
        f"Type match (move vs switch): {metrics['type_match']}/{t} ({100.0 * metrics['type_match'] / t:.1f}%)"
    )
    print(
        f"Exact match (full action): {metrics['exact_match']}/{t} ({100.0 * metrics['exact_match'] / t:.1f}%)"
    )
    print(f"{'=' * 60}")


def save_metrics_json(
    path: str,
    metrics: dict[str, int],
    *,
    extra: dict[str, Any] | None = None,
) -> None:
    out = dict(metrics)
    if extra:
        out["meta"] = extra
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

print('Showdown eval helpers defined')


### 3b — Validate an action against the demo log

The agent must output one line: `move <Name>` or `switch <Species>`.


In [ ]:
demo_log = tutorial_demo_log_with_request()
print(f"demo_log_chars={len(demo_log)}")

legal_move = validate_action_against_log("move Brave Bird", demo_log)
wrong_move = validate_action_against_log("move Earthquake", demo_log)  # format OK, not in |request|
bad_format = validate_action_against_log("hello world", demo_log)
print("legal move:", legal_move)
print("wrong move:", wrong_move)
print("bad format:", bad_format)


### 3c — Parse protocol lines

Showdown logs use pipe-separated lines such as `|move|p2a: Pikachu|Thunderbolt|...`.


In [ ]:
sample_line = "|move|p2a: Pikachu|Thunderbolt|p1a: Charizard"
print("fields:", showdown_fields(sample_line))

# demo_log is mid-battle (no |win| yet) — used for inference in Steps 8+
winner_side, winner_name = extract_winner_side(demo_log)
print(f"winner from demo_log: side={winner_side!r} name={winner_name!r}")


### 3d — Local Showdown server helpers

Clones upstream [smogon/pokemon-showdown](https://github.com/smogon/pokemon-showdown) to `/tmp/pokemon-showdown` on first use, runs `npm install`, and starts a private server on port **8000**.


In [ ]:
# Local Showdown server helpers
"""Showdown connectivity helpers: guest auth patch, local server, unique accounts."""
from __future__ import annotations

import argparse
import json
import os
import re
import shutil
import socket
import subprocess
import sys
import time
import uuid
from pathlib import Path
from typing import Any

import requests
from poke_env.ps_client import ps_client
from poke_env.ps_client.account_configuration import AccountConfiguration
from poke_env.ps_client.server_configuration import (
    LocalhostServerConfiguration,
    ShowdownServerConfiguration,
)

PS_DIR = os.environ.get("POKE_SHOWDOWN_DIR", "/tmp/pokemon-showdown")
PS_PORT = int(os.environ.get("POKE_SHOWDOWN_PORT", "8000"))
PS_LOG = os.environ.get("POKE_SHOWDOWN_LOG", "/tmp/ps.log")
PS_GIT = os.environ.get(
    "POKE_SHOWDOWN_GIT", "https://github.com/smogon/pokemon-showdown.git"
)

_GUEST_AUTH_PATCHED = False


def patch_guest_authentication() -> None:
    """Official PS requires getassertion; poke-env 0.15 sends empty assertion for guests."""
    global _GUEST_AUTH_PATCHED
    if _GUEST_AUTH_PATCHED:
        return

    original = ps_client.PSClient.log_in

    async def log_in(self, split_message: list[str]) -> None:
        if self.account_configuration.password:
            log_in_request = requests.post(
                self.server_configuration.authentication_url,
                data={
                    "act": "login",
                    "name": self.account_configuration.username,
                    "pass": self.account_configuration.password,
                    "challstr": split_message[2] + "%7C" + split_message[3],
                },
                timeout=10.0,
            )
            self.logger.info("Sending authentication request")
            assertion = json.loads(log_in_request.text[1:])["assertion"]
        elif "play.pokemonshowdown.com" in self.server_configuration.authentication_url:
            self.logger.info("Guest getassertion request")
            resp = requests.post(
                self.server_configuration.authentication_url,
                data={
                    "act": "getassertion",
                    "userid": self.account_configuration.username,
                    "challstr": split_message[2] + "%7C" + split_message[3],
                },
                timeout=10.0,
            )
            assertion = resp.text.strip()
            if assertion.startswith(";") or assertion.startswith("]"):
                assertion = ""
        else:
            self.logger.info("Bypassing authentication request")
            assertion = ""

        await self.send_message(f"/trn {self.username},0,{assertion}")
        if self._avatar is not None:
            await self.change_avatar(self._avatar)

    ps_client.PSClient.log_in = log_in  # type: ignore[method-assign]
    _GUEST_AUTH_PATCHED = True


def port_open(host: str, port: int, timeout: float = 1.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        try:
            sock.connect((host, port))
            return True
        except OSError:
            return False


def resolve_node_bin() -> str:
    """Node.js 22+ is required by upstream pokemon-showdown."""
    env = os.environ.get("POKE_NODE_BIN", "").strip()
    candidates = [env, "/opt/node22/bin/node", shutil.which("node") or ""]
    seen: set[str] = set()
    for cand in candidates:
        if not cand or cand in seen:
            continue
        seen.add(cand)
        if not Path(cand).is_file():
            continue
        try:
            out = subprocess.run(
                [cand, "--version"],
                capture_output=True,
                text=True,
                timeout=5.0,
                check=False,
            )
            major = int(re.match(r"v?(\d+)", out.stdout.strip() or "0").group(1))
            if major >= 22:
                return cand
        except (ValueError, subprocess.SubprocessError):
            continue
    raise RuntimeError("Node.js 22+ required for local Showdown (install nodesource 22.x).")


def ensure_showdown_repo(ps_dir: str = PS_DIR) -> Path:
    ps_path = Path(ps_dir)
    if (ps_path / "package.json").is_file():
        return ps_path
    ps_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Cloning pokemon-showdown -> {ps_path} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", PS_GIT, str(ps_path)],
        check=True,
    )
    return ps_path


def ensure_showdown_deps(ps_path: Path, *, node_bin: str | None = None) -> None:
    if (ps_path / "node_modules").is_dir():
        return
    npm = shutil.which("npm") or "npm"
    print(f"Running npm install in {ps_path} (first time may take a few minutes) ...")
    subprocess.run(
        [npm, "install", "--no-audit", "--no-fund"],
        cwd=str(ps_path),
        check=True,
        env={**os.environ, "PATH": os.environ.get("PATH", "")},
    )
    _ = node_bin


def websocket_ready(host: str = "127.0.0.1", port: int = PS_PORT, timeout: float = 3.0) -> bool:
    try:
        import http.client

        conn = http.client.HTTPConnection(host, port, timeout=timeout)
        conn.request(
            "GET",
            "/showdown/websocket",
            headers={
                "Connection": "Upgrade",
                "Upgrade": "websocket",
                "Sec-WebSocket-Version": "13",
                "Sec-WebSocket-Key": "dGhlIHNhbXBsZSBub25jZQ==",
            },
        )
        resp = conn.getresponse()
        conn.close()
        return resp.status == 101
    except OSError:
        return False


def showdown_public_url(port: int = PS_PORT) -> str:
    custom = os.environ.get("POKE_SHOWDOWN_PUBLIC_URL", "").strip()
    if custom:
        return custom.rstrip("/")
    host = os.environ.get("POKE_SHOWDOWN_PUBLIC_HOST", "127.0.0.1").strip() or "127.0.0.1"
    return f"http://{host}:{port}"


EVAL_OUTPUT_NOISE = (
    "Task was destroyed but it is pending",
    "task: <Task pending",
    "[asyncio|ERROR]",
    "🦥 Unsloth:",
    "Unsloth Zoo",
    "==((====))==",
    "UserWarning:",
    "expandable_segments",
    "pip is available",
    "[notice]",
)


def is_noisy_eval_line(line: str) -> bool:
    return any(marker in line for marker in EVAL_OUTPUT_NOISE)


class _FilteredEvalStream:
    def __init__(self, stream: Any) -> None:
        self._stream = stream

    def write(self, s: str) -> None:
        if not s or is_noisy_eval_line(s):
            return
        self._stream.write(s)

    def flush(self) -> None:
        self._stream.flush()

    def __getattr__(self, name: str) -> Any:
        return getattr(self._stream, name)


def quiet_eval_logs(*, force: bool = False) -> None:
    if not force and os.environ.get("POKEMON_EVAL_QUIET", "1") == "0":
        return
    import logging
    import warnings

    warnings.filterwarnings("ignore", category=UserWarning)
    warnings.filterwarnings("ignore", category=FutureWarning)
    os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
    os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

    class _HideAsyncioCleanup(logging.Filter):
        def filter(self, record: logging.LogRecord) -> bool:
            msg = record.getMessage()
            return "Task was destroyed" not in msg and not msg.strip().startswith(
                "task: <Task pending"
            )

    flt = _HideAsyncioCleanup()
    for name in ("asyncio", "transformers", "unsloth", "unsloth_zoo", "poke_env"):
        lg = logging.getLogger(name)
        lg.setLevel(logging.ERROR)
        lg.addFilter(flt)
    if not isinstance(sys.stdout, _FilteredEvalStream):
        sys.stdout = _FilteredEvalStream(sys.__stdout__)
    if not isinstance(sys.stderr, _FilteredEvalStream):
        sys.stderr = _FilteredEvalStream(sys.__stderr__)


def print_showdown_browser_help(port: int = PS_PORT) -> None:
    local_port = os.environ.get("POKE_SHOWDOWN_LOCAL_PORT", str(port)).strip() or str(port)
    testclient = (
        f"https://play.pokemonshowdown.com/testclient-new.html?~~127.0.0.1:{local_port}"
    )
    print("\n=== Pokemon Showdown (local, private server) ===")
    print(f"  Browser (SSH / port-forward): {testclient}")
    print("  Do NOT use the psim.us redirect page for remote access.")
    print("  Step 15: challenge the friendly agent in gen9randombattle.")
    if os.environ.get("SSH_CONNECTION"):
        print(
            f"\n  Example: ssh -L {local_port}:127.0.0.1:{port} user@host"
            f"  then open the testclient URL above."
        )


def ensure_local_showdown(
    port: int = PS_PORT,
    ps_dir: str = PS_DIR,
    node_bin: str | None = None,
) -> str:
    if port_open("127.0.0.1", port) and websocket_ready("127.0.0.1", port):
        return f"already listening on {port} (websocket OK)"

    node = node_bin or resolve_node_bin()
    ps_path = ensure_showdown_repo(ps_dir)
    ensure_showdown_deps(ps_path, node_bin=node)

    if port_open("127.0.0.1", port) and not websocket_ready("127.0.0.1", port):
        raise RuntimeError(
            f"Port {port} is in use but Showdown websocket failed. "
            f"Stop the other process or set POKE_SHOWDOWN_PORT."
        )

    if not port_open("127.0.0.1", port):
        log_f = open(PS_LOG, "a", encoding="utf-8")
        subprocess.Popen(
            [node, "pokemon-showdown", "start", "--no-security", str(port)],
            cwd=str(ps_path),
            stdout=log_f,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    deadline = time.monotonic() + 90.0
    while time.monotonic() < deadline:
        if port_open("127.0.0.1", port, timeout=0.5) and websocket_ready("127.0.0.1", port):
            return f"started on {port} (websocket OK)"
        time.sleep(0.5)
    raise RuntimeError(f"Showdown did not become ready on {port} within 90s; see {PS_LOG}")


def bootstrap_local_showdown(**kwargs: Any) -> str:
    patch_guest_authentication()
    status = ensure_local_showdown(**kwargs)
    print_showdown_browser_help(kwargs.get("port", PS_PORT))
    return status


def server_configuration(name: str | None = None):
    mode = (
        name
        or os.environ.get("POKE_SHOWDOWN_SERVER")
        or os.environ.get("POKEMON_SHOWDOWN_SERVER", "local")
    ).strip().lower()
    if mode == "official":
        return ShowdownServerConfiguration
    return LocalhostServerConfiguration


def resolve_friendly_account(name: str) -> AccountConfiguration:
    """Pick a guest username; append a suffix by default to avoid |nametaken| on re-run."""
    if os.environ.get("POKEMON_FRIENDLY_UNIQUE", "1").strip().lower() in ("0", "false", "no"):
        return AccountConfiguration(name, None)
    base = re.sub(r"[^A-Za-z0-9]", "", name)[:12] or "LLMAgent"
    uname = f"{base}{uuid.uuid4().hex[:6]}"[:18]
    return AccountConfiguration(uname, None)


def unique_account(prefix: str) -> AccountConfiguration:
    uname = f"{prefix}{uuid.uuid4().hex[:10]}"[:18]
    return AccountConfiguration(uname, None)


def player_kwargs(name_key: str, *, server: str | None = None) -> dict[str, Any]:
    return {
        "max_concurrent_battles": 1,
        "server_configuration": server_configuration(server),
        "account_configuration": unique_account(name_key),
        "avatar": None,
    }


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Start local Pokemon Showdown")
    parser.add_argument("--port", type=int, default=PS_PORT)
    parser.add_argument("--ps-dir", default=PS_DIR)
    args = parser.parse_args(argv)
    try:
        print(bootstrap_local_showdown(port=args.port, ps_dir=args.ps_dir))
        return 0
    except Exception as exc:
        print(f"ERROR: {exc}", file=sys.stderr)
        return 1


from poke_env.ps_client.server_configuration import LocalhostServerConfiguration
print('Showdown server helpers defined')


## Step 4 — Learn Pokémon Showdown rules

Each turn the agent outputs **one line**: `move <Name>` or `switch <Species>` (optional `terastallize`).

| Concept | Battle UI | Protocol |
|---------|-----------|----------|
| Active / bench | sprites + party | `|switch|p2a: Pikachu` |
| Move | animation | `|move|p2a: Pikachu|Thunderbolt|p1a: Charizard` |
| Tera | crystal overlay | `|terastallize|p2a: Pikachu|Electric` |

The next cells show species sprites and an interactive mini demo.

> **Server note:** Leave `POKEMON_USE_POKE_ENV=0` (default). Steps 14–15 use a **local** Showdown server on port 8000.


In [ ]:
from IPython.display import HTML, display

# Showdown sprites (gen5 CDN) + offline emoji fallback
SHOWDOWN_SPRITE = "https://play.pokemonshowdown.com/sprites/gen5/{name}.png"
CARDS = [
    ("Pikachu", "electric", "#F8D030", "⚡"),
    ("Charizard", "fire", "#F08030", "🔥"),
    ("Garchomp", "dragon", "#7038F8", "🐉"),
    ("Corviknight", "flying", "#A890F0", "🦅"),
    ("Heatran", "fire", "#F08030", "🌋"),
    ("Landorus", "ground", "#E0C068", "🌍"),
    ("Dragapult", "dragon", "#7038F8", "👻"),
    ("Great Tusk", "ground", "#E0C068", "🦣"),
]

html = '<div style="display:flex;gap:10px;flex-wrap:wrap">'
for name, typ, color, emoji in CARDS:
    slug = name.upper().replace(" ", "").replace("-", "")
    img = SHOWDOWN_SPRITE.format(name=slug)
    html += (
        f'<div style="width:140px;padding:8px;border:2px solid {color};border-radius:8px;text-align:center;background:#fafafa">'
        f'<img src="{img}" alt="{name}" style="width:96px;height:96px;image-rendering:pixelated" '
        f'onerror="this.style.display=\'none\';this.nextElementSibling.style.display=\'block\'">'
        f'<div style="font-size:48px;display:none">{emoji}</div>'
        f'<b>{name}</b><br><span style="color:{color};font-size:12px">{typ}</span>'
        f'<br><code style="font-size:10px">|switch|p2a: {name}</code></div>'
    )
html += "</div>"

display(HTML(html))
display(HTML(
    '<p><b>Protocol mapping:</b>'
    '<code>|move|p2a: Pikachu|Thunderbolt|p1a: Charizard</code> → output <code>move Thunderbolt</code><br>'
    '<code>|switch|p2a: Garchomp|Garchomp|100/100</code> → output <code>switch Garchomp</code><br>'
    '<a href="https://play.pokemonshowdown.com/" target="_blank">▶ Open official Showdown</a></p>'
))
print(f"Rendered {len(CARDS)} species cards (Showdown CDN + offline fallback).")


In [ ]:
import json
import os
import urllib.error
import urllib.request

USE_POKEAPI = os.environ.get("POKEMON_USE_POKEAPI", "0").strip() == "1"
USE_POKE_ENV = os.environ.get("POKEMON_USE_POKE_ENV", "0").strip() == "1"

FALLBACK_POKEMON = {
    "pikachu": {"name": "pikachu", "types": ["electric"], "hp": 35},
    "garchomp": {"name": "garchomp", "types": ["dragon", "ground"], "hp": 108},
    "charizard": {"name": "charizard", "types": ["fire", "flying"], "hp": 78},
    "blastoise": {"name": "blastoise", "types": ["water"], "hp": 79},
}


def pokeapi_get(path: str) -> dict:
    url = f"https://pokeapi.co/api/v2/{path.strip('/')}/"
    req = urllib.request.Request(url, headers={"User-Agent": "pokemon-rocm-tutorial/1.0"})
    with urllib.request.urlopen(req, timeout=8) as resp:
        return json.load(resp)


def pokemon_brief(name: str) -> dict:
    key = name.lower()
    if key not in FALLBACK_POKEMON:
        raise KeyError(f"unknown species {key}")
    if not USE_POKEAPI:
        return {**FALLBACK_POKEMON[key], "source": "builtin"}
    try:
        data = pokeapi_get(f"pokemon/{key}")
        types = [t["type"]["name"] for t in data["types"]]
        hp = next(s["base_stat"] for s in data["stats"] if s["stat"]["name"] == "hp")
        return {"name": data["name"], "types": types, "hp": hp, "source": "pokeapi"}
    except Exception as exc:
        print(f"[pokeapi] fallback for {key}: {type(exc).__name__}")
        return {**FALLBACK_POKEMON[key], "source": "fallback"}


TYPE_CHART = {
    ("electric", "water"): 2.0, ("electric", "ground"): 0.0,
    ("fire", "grass"): 2.0, ("water", "fire"): 2.0,
    ("ground", "electric"): 2.0, ("dragon", "dragon"): 2.0,
}


def type_multiplier(move_type: str, defender_types: list) -> float:
    m = 1.0
    for dt in defender_types:
        m *= TYPE_CHART.get((move_type, dt), 1.0)
    return m


class MiniShowdownDemo:
    def __init__(self, p2_names=("pikachu", "garchomp"), p1_names=("charizard", "blastoise")):
        self.p1 = [pokemon_brief(n) for n in p1_names]
        self.p2 = [pokemon_brief(n) for n in p2_names]
        self.p1_active, self.p2_active = 0, 0
        self.p1_hp = [mon["hp"] for mon in self.p1]
        self.p2_hp = [mon["hp"] for mon in self.p2]
        self.turn, self.log = 1, []

    def _active(self, side):
        mons = self.p1 if side == "p1" else self.p2
        idx = self.p1_active if side == "p1" else self.p2_active
        return mons[idx], idx

    def print_state(self):
        p1, _ = self._active("p1")
        p2, _ = self._active("p2")
        print(f"\n=== Turn {self.turn} ===")
        print(f"p1a: {p1['name']} HP={self.p1_hp[self.p1_active]} types={p1['types']} ({p1['source']})")
        print(f"p2a: {p2['name']} HP={self.p2_hp[self.p2_active]} types={p2['types']} ({p2['source']})")
        print("Bench p2:", [m["name"] for i, m in enumerate(self.p2) if i != self.p2_active])

    def apply_action(self, cmd: str) -> bool:
        cmd = cmd.strip().lower()
        if cmd.startswith("switch "):
            target = cmd.split(" ", 1)[1].strip().lower()
            for i, mon in enumerate(self.p2):
                if mon["name"].lower() == target and i != self.p2_active and self.p2_hp[i] > 0:
                    self.log.append(f"|switch|p2a: {mon['name'].title()}")
                    self.p2_active = i
                    return True
            print("Illegal switch — use a benched species name.")
            return False
        if cmd.startswith("move "):
            move_name = cmd.split(" ", 1)[1]
            p1, _ = self._active("p1")
            p2, _ = self._active("p2")
            mt = "electric" if "thunder" in move_name else "ground" if "earth" in move_name else p2["types"][0]
            dmg = max(1, int(20 * type_multiplier(mt, p1["types"])))
            self.p1_hp[self.p1_active] = max(0, self.p1_hp[self.p1_active] - dmg)
            self.log.append(f"|move|p2a: {p2['name'].title()}|{move_name.title()}|p1a: {p1['name'].title()}")
            print(f"Dealt ~{dmg} damage.")
            return True
        print("Use: move <name>  OR  switch <species>")
        return False

    def play_demo_round(self):
        self.print_state()
        for cmd in ("move thunderbolt", "switch garchomp", "move earthquake"):
            print(f"\n> p2: {cmd}")
            if not self.apply_action(cmd):
                raise RuntimeError(f"demo action failed: {cmd}")
            self.turn += 1
            self.print_state()
        print("\n--- protocol tail ---")
        for line in self.log[-4:]:
            print(line)


print("=== Auto demo turn ===")
_demo = MiniShowdownDemo()
_demo.play_demo_round()
print("Step 4 auto demo OK")

# --- Interactive: type your command ---
try:
    import ipywidgets as widgets
    from IPython.display import display

    _interactive = MiniShowdownDemo()
    cmd_box = widgets.Text(value="move thunderbolt", description="p2 cmd:", layout=widgets.Layout(width="420px"))
    out = widgets.Output()

    def on_go(_):
        with out:
            out.clear_output()
            ok = _interactive.apply_action(cmd_box.value)
            if ok:
                _interactive.turn += 1
            _interactive.print_state()
            print("log:", _interactive.log[-1] if _interactive.log else "(empty)")

    btn = widgets.Button(description="Run turn", button_style="primary")
    btn.on_click(on_go)
    display(widgets.VBox([
        widgets.HTML("<b>Interactive demo</b> — 输入 <code>move thunderbolt</code> 或 <code>switch garchomp</code>"),
        widgets.HBox([cmd_box, btn]), out,
    ]))
except ImportError:
    print("[ipywidgets] ipywidgets not installed — auto demo only")

# --- Optional: official Showdown ladder (off by default) ---
# Step 14–15 use a LOCAL Showdown server — do not need this block.
# POKEMON_USE_POKE_ENV=1 connects to play.pokemonshowdown.com and may hang without a login/token.
if USE_POKE_ENV:
    import asyncio
    from poke_env.player import RandomPlayer
    from poke_env.ps_client.server_configuration import ShowdownServerConfiguration

    _ladder_timeout = float(os.environ.get("POKEMON_POKE_ENV_TIMEOUT", "20"))

    async def _ladder():
        p = RandomPlayer(server_configuration=ShowdownServerConfiguration, max_concurrent_battles=1)
        try:
            await asyncio.wait_for(p.ladder(1), timeout=_ladder_timeout)
        finally:
            await p.ps_client.stop_listening()

    print(f"\n[poke-env] Optional official ladder ({_ladder_timeout:.0f}s timeout)...")
    try:
        asyncio.get_event_loop().run_until_complete(_ladder())
        print("[poke-env] Ladder demo finished.")
    except asyncio.TimeoutError:
        print("[poke-env] Timed out — skip on servers without Showdown login. Use Step 14 local Showdown.")
    except Exception as exc:
        print(f"[poke-env] Skipped official ladder: {exc}")
else:
    print("\n[poke-env] Skipped (default). Battle eval uses local Showdown in Step 14.")


<a id="step5"></a>

## Step 5 — Phase 1: Pick a base model (same battle, different brains)


<div style="text-align:center; padding:12px; background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%); border-radius:12px; margin:12px 0;">
  <img src="https://play.pokemonshowdown.com/sprites/gen5ani/garchomp.gif" width="96" alt="Garchomp"/>
  <img src="https://play.pokemonshowdown.com/sprites/gen5ani/corviknight.gif" width="96" alt="Corviknight"/>
  <img src="https://play.pokemonshowdown.com/sprites/gen5ani/pikachu.gif" width="96" alt="Pikachu"/>
  <p style="color:#e94560; font-weight:bold; margin:8px 0 0;">Pokémon Showdown — teach an LLM to pick <code>move</code> or <code>switch</code></p>
</div>


**Goal:** Feed one **real Showdown turn** to several checkpoints **without fine-tuning**. You should see base models ramble or leak `thinking` — fine-tuned agents emit `move …` / `switch …`.

### Why we chose **Qwen3-4B** for this tutorial

| Criterion | Qwen3-4B |
|-----------|----------|
| Instruction following | Strong chat template; easy `move`/`switch` format |
| ROCm + Unsloth | Official Unsloth path on **MI300X bf16** (no 4-bit train instability) |
| VRAM | ~24 GB for LoRA demo; scales to 48 GB for full SFT |
| License | Apache-2.0 friendly for Hub publishing |
| Pokémon task | 4B enough for single-turn tactics; 7B+ optional for ladder |

### Two comparison axes (set `POKEMON_COMPARE`)

| Mode | Axis | Question | Models |
|------|------|----------|--------|
| **`depth`** (default) | Training depth | Same Qwen3-4B — how far does training go? | base → tutorial SFT → tutorial GRPO → full SFT → battle GRPO |
| **`backbone`** | Base model | Zero-shot — which backbone follows `move/switch` best? | Qwen3-4B, Qwen3-1.7B, Qwen2.5-3B |
| **`both`** | Both | Run depth + backbone tables | All of the above |

Set `POKEMON_RUN_MODEL_COMPARE=0` to skip when VRAM is tight. Override backbones: `POKEMON_BACKBONE_MODELS=qwen3_4b:Qwen/Qwen3-4B:qwen3,...`


In [ ]:
import gc
import os
from pathlib import Path

import torch

print("step5_model_compare")

USER = tutorial_demo_log_with_request()
assert "|request|" in USER or "|turn|" in USER
print(f"demo_log_chars={len(USER)}")
comparison_rows = []

RUN_MODEL_COMPARE = os.environ.get("POKEMON_RUN_MODEL_COMPARE", "1").strip() != "0"
COMPARE_MODE = os.environ.get("POKEMON_COMPARE", "depth").strip().lower()
if COMPARE_MODE not in {"depth", "backbone", "both"}:
    raise ValueError(f"POKEMON_COMPARE must be depth|backbone|both, got {COMPARE_MODE!r}")
MODEL_ROOT = Path(os.environ.get("POKEMON_MODEL_ROOT", str(WORK_ROOT)))
BASE_MODEL = "Qwen/Qwen3-4B"
HF_SFT = os.environ.get("POKEMON_HF_SFT_MODEL", "GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-sft")
HF_GRPO = os.environ.get("POKEMON_HF_GRPO_MODEL", "GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-grpo")
HF_FULL_SFT = "GoldenGrapeGentleman1/pokemon-showdown-agent-full-sft"


def _parse_backbone_specs():
    raw = os.environ.get("POKEMON_BACKBONE_MODELS", "").strip()
    if raw:
        specs = []
        for part in raw.split(","):
            label, mid, chat = (x.strip() for x in part.split(":", 2))
            specs.append((label, mid, chat))
        return specs
    return [
        ("backbone_qwen3_4b", "Qwen/Qwen3-4B", "qwen3"),
        ("backbone_qwen3_1_7b", "Qwen/Qwen3-1.7B", "qwen3"),
        ("backbone_qwen2_5_3b", "Qwen/Qwen2.5-3B", "qwen-2.5"),
    ]


def build_compare_plan(mode: str, model_root: Path) -> list[dict]:
    plan: list[dict] = []

    if mode in {"depth", "both"}:
        depth_chain = [
            ("depth_base", BASE_MODEL, "full", "qwen3", "0 step"),
            ("depth_tutorial_sft", HF_SFT, "adapter", "qwen3", "50 step SFT"),
            ("depth_tutorial_grpo", HF_GRPO, "adapter", "qwen3", "Tutorial GRPO"),
            ("depth_full_sft", HF_FULL_SFT, "full", "qwen3", "Full replay SFT (Hub)"),
        ]
        _full_sft_env = os.environ.get("POKEMON_FULL_SFT_DIR", "").strip()
        local_full_sft = Path(_full_sft_env) if _full_sft_env else None
        if local_full_sft is None or not local_full_sft.is_dir():
            for cand in sorted(model_root.glob("model_output_*/merged_model")):
                if (cand / "config.json").exists():
                    local_full_sft = cand
                    break
        if local_full_sft is not None and local_full_sft.is_dir() and (local_full_sft / "config.json").exists():
            depth_chain[-1] = ("depth_full_sft_local", str(local_full_sft), "full", "qwen3", "Full replay SFT (local)")
        local_sft = model_root / "outputs/pokemon_showdown_agent_tutorial/checkpoint-50"
        if local_sft.is_dir() and (local_sft / "adapter_config.json").exists():
            depth_chain[1] = ("depth_tutorial_sft_local", str(local_sft), "adapter", "qwen3", "50 step SFT (local)")
        local_grpo = None
        _tutorial_grpo_dir = model_root / "grpo_outputs"
        if _tutorial_grpo_dir.is_dir():
            for ckpt in sorted(_tutorial_grpo_dir.glob("checkpoint-*"), reverse=True):
                if (ckpt / "adapter_config.json").is_file():
                    local_grpo = ckpt
                    break
        if local_grpo is not None and local_grpo.is_dir():
            depth_chain[2] = ("depth_tutorial_grpo_local", str(local_grpo), "adapter", "qwen3", "Tutorial GRPO (local)")
        local_battle_grpo = None
        _grpo_env = os.environ.get("POKEMON_GRPO_CKPT", "").strip()
        if _grpo_env and Path(_grpo_env).is_dir():
            local_battle_grpo = Path(_grpo_env)
        else:
            for parent in sorted(model_root.glob("grpo_*_outputs")):
                for ckpt in sorted(parent.glob("checkpoint-*"), reverse=True):
                    if (ckpt / "adapter_config.json").is_file():
                        local_battle_grpo = ckpt
                        break
                if local_battle_grpo is not None:
                    break
        if local_battle_grpo is not None and local_battle_grpo.is_dir():
            depth_chain.append(("depth_battle_grpo", str(local_battle_grpo), "adapter", "qwen3", "Battle GRPO"))
        for label, mid, kind, chat, stage in depth_chain:
            plan.append({"axis": "depth", "label": label, "model_id": mid, "kind": kind, "chat": chat, "stage": stage})

    if mode in {"backbone", "both"}:
        for label, mid, chat in _parse_backbone_specs():
            plan.append({"axis": "backbone", "label": label, "model_id": mid, "kind": "full", "chat": chat, "stage": "0 step (zero-shot)"})

    return plan


COMPARE_PLAN = build_compare_plan(COMPARE_MODE, MODEL_ROOT)
print(f"POKEMON_COMPARE={COMPARE_MODE}  models={len(COMPARE_PLAN)}")
for item in COMPARE_PLAN:
    print(f"  [{item['axis']}] {item['label']} ({item['stage']}) -> {item['model_id']}")

if not torch.cuda.is_available():
    print("step5_skipped: no GPU")
elif not RUN_MODEL_COMPARE:
    print("step5_skipped: set POKEMON_RUN_MODEL_COMPARE=1 to enable")
elif not COMPARE_PLAN:
    print("step5_skipped: empty compare plan")
else:
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    SYS = (
        "You are a Pokemon Showdown battle AI. You play as p2. "
        "Given the battle log, output your next action. "
        "Format: move <name> OR switch <name>. "
        "Append terastallize if you terastallize this turn."
    )

    def apply_prompt(tok, msgs, chat: str):
        kwargs = dict(tokenize=False, add_generation_prompt=True)
        if chat == "qwen3":
            kwargs["enable_thinking"] = False
        try:
            return tok.apply_chat_template(msgs, **kwargs)
        except TypeError:
            kwargs.pop("enable_thinking", None)
            return tok.apply_chat_template(msgs, **kwargs)

    def predict(m, tok, label, chat: str):
        msgs = [{"role": "system", "content": SYS}, {"role": "user", "content": USER}]
        text = apply_prompt(tok, msgs, chat)
        inp = tok(text, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = m.generate(
                **inp,
                max_new_tokens=48,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
                use_cache=False,
            )
        raw = tok.decode(out[0][inp.input_ids.shape[-1]:], skip_special_tokens=True)
        action = postprocess_agent_response(raw).splitlines()[0].strip()
        print(f"[{label}] {action}")
        return action

    def load_model(item):
        label, mid, kind, chat = item["label"], item["model_id"], item["kind"], item["chat"]
        if kind == "full":
            m, tok = FastLanguageModel.from_pretrained(
                model_name=mid,
                max_seq_length=2048,
                dtype=torch.bfloat16,
                load_in_4bit=False,
                attn_implementation="sdpa",
            )
        else:
            m, tok = FastLanguageModel.from_pretrained(
                model_name=BASE_MODEL,
                max_seq_length=2048,
                dtype=torch.bfloat16,
                load_in_4bit=False,
                attn_implementation="sdpa",
            )
            m.load_adapter(mid, adapter_name=label[:20])
            m.set_adapter(label[:20])
        tok = get_chat_template(tok, chat_template=chat)
        FastLanguageModel.for_inference(m)
        return m, tok

    for item in COMPARE_PLAN:
        m = tok = None
        label = item["label"]
        try:
            print(f"\n[{item['axis']}] Loading {item['model_id']} ({item['kind']})...")
            m, tok = load_model(item)
            action = predict(m, tok, label, item["chat"])
            comparison_rows.append({**item, "action": action})
        except Exception as exc:
            print(f"[{label}] FAILED: {exc}")
            comparison_rows.append({**item, "action": f"ERROR: {exc}"})
        finally:
            if m is not None:
                del m
            if tok is not None:
                del tok
            gc.collect()
            torch.cuda.empty_cache()


def _print_table(axis: str, title: str) -> None:
    rows = [r for r in comparison_rows if r.get("axis") == axis]
    print(f"\n### {title}")
    print("| checkpoint | stage | model | action |")
    if rows:
        for row in rows:
            print(f"| {row['label']} | {row['stage']} | `{row['model_id']}` | `{row['action']}` |")
    else:
        print(f"| (none) | — | — | axis={axis} not run |")


if COMPARE_MODE in {"depth", "both"}:
    _print_table("depth", "Axis A — training depth (Qwen3-4B)")
if COMPARE_MODE in {"backbone", "both"}:
    _print_table("backbone", "Axis B — backbone (zero-shot)")
if not comparison_rows:
    print("\n| (skipped) | — | enable GPU + POKEMON_RUN_MODEL_COMPARE=1 |")
print("\nStep 5 OK — Step 7 reloads base Qwen for training.")


## Step 6 — Data processing walkthrough (long log → training row)

### 6a — Inspect one raw replay

### Phase 2 — Data pipeline (before any SFT)

Long Showdown replays are **truncated** to a mid-battle prefix (~35–40 lines) so the model sees a realistic decision point without blowing the 2048 token budget. Below: raw log → turn slice → **chat training row**.


In [ ]:

from datasets import load_dataset
row = next(iter(load_dataset("milkkarten/pokemon-showdown-replays-merged", split="train", streaming=True)))
log = row["log"]
print(f"chars={len(log)} lines={len(log.splitlines())}")
print("--- head ---\n" + "\n".join(log.splitlines()[:10]))
print("--- tail ---\n" + "\n".join(log.splitlines()[-6:]))
_raw_log = log


### 6b — Parse protocol + winner side


In [ ]:
winner_side, winner_name = extract_winner_side(_raw_log)
print(f"winner_side={winner_side} winner_name={winner_name}")


### 6c — Slice one turn (long → short prefix)


In [ ]:

from IPython.display import HTML, display

MAX_LOG_CHARS = 6000
lines = _raw_log.strip().split("\n")
turns = [(int(f[1]), i) for i, ln in enumerate(lines) for f in [showdown_fields(ln)] if len(f) >= 2 and f[0] == "turn"]
idx = 0 if len(turns) == 2 else 1
_, tline = turns[idx]
nline = turns[idx + 1][1] if idx + 1 < len(turns) else len(lines)
action = None
for j in range(tline + 1, nline):
    f = showdown_fields(lines[j])
    if len(f) < 3: continue
    if f[0] == "move" and f[1].startswith(f"{winner_side}a:"): action = f"move {f[2]}"; break
    if f[0] == "switch" and f[1].startswith(f"{winner_side}a:"):
        action = f"switch {f[1].split(': ',1)[-1]}"; break
log_prefix = "\n".join(lines[: tline + 1])

# Visualize: long log -> short prefix
display(HTML(f"""
<div style="display:flex;gap:16px;font-family:monospace;font-size:12px">
  <div style="flex:1;padding:12px;background:#fff3f3;border:1px solid #fcc;border-radius:8px">
    <b>Raw log</b><br>{len(_raw_log):,} chars · {len(lines)} lines
    <pre style="max-height:180px;overflow:auto;margin-top:8px">{chr(10).join(lines[:6])}
...
{chr(10).join(lines[-3:])}</pre>
  </div>
  <div style="flex:0 0 40px;text-align:center;font-size:24px;padding-top:60px">→</div>
  <div style="flex:1;padding:12px;background:#f3fff3;border:1px solid #cfc;border-radius:8px">
    <b>Training prefix</b><br>{len(log_prefix):,} chars · fits={len(log_prefix) <= MAX_LOG_CHARS}
    <pre style="max-height:180px;overflow:auto;margin-top:8px">{chr(10).join(log_prefix.splitlines()[-8:])}</pre>
    <b style="color:green">assistant: {action}</b>
  </div>
</div>
"""))
print(f"action={action} prefix_len={len(log_prefix)}")


### 6d — Chat row structure


In [ ]:
for role, body in [
    ("system", SYSTEM_TEMPLATE.format(side=winner_side)),
    ("user", log_prefix[:120] + "..."),
    ("assistant", action),
]:
    print(f"{role}: {body}")


## Step 7 — Load Qwen3-4B and attach LoRA adapters

We load [`Qwen/Qwen3-4B`](https://huggingface.co/Qwen/Qwen3-4B) and attach **LoRA**. Keep **`load_in_4bit=False`** and **`attn_implementation="sdpa"`** for ROCm stability.


In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

max_seq_length = 2048
dtype = torch.bfloat16
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    attn_implementation="sdpa",
)

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"trainable_params={trainable_params / 1e6:.1f}M")
print(f"trainable_ratio={100 * trainable_params / all_params:.2f}%")


## Step 8 — Stream and format replay logs (batch / disk-safe)

Same logic as Step 6 in streaming mode. Only **`MAX_TRAIN_SAMPLES`** are materialized in memory.


In [ ]:
from datasets import Dataset, load_dataset

MIN_RATING = 1400
MAX_TRAIN_SAMPLES = 2000
SHUFFLE_BUFFER = 10_000
MAX_LOG_CHARS = 6000

SYSTEM_TEMPLATE = (
    "You are a Pokemon Showdown battle AI. You play as {side}. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)



def format_sample(example):
    log_text = example["log"]
    side, _winner_name = extract_winner_side(log_text)
    if not side:
        return {"text": ""}

    lines = log_text.strip().split("\n")
    turn_positions = []
    for i, line in enumerate(lines):
        f = showdown_fields(line)
        if len(f) >= 2 and f[0] == "turn":
            try:
                turn_positions.append((int(f[1]), i))
            except ValueError:
                pass

    if len(turn_positions) < 2:
        return {"text": ""}

    target_turn_idx = 0 if len(turn_positions) == 2 else 1
    _, turn_line_idx = turn_positions[target_turn_idx]
    next_turn_line = turn_positions[target_turn_idx + 1][1] if target_turn_idx + 1 < len(turn_positions) else len(lines)

    action = None
    for j in range(turn_line_idx + 1, next_turn_line):
        f = showdown_fields(lines[j])
        if len(f) < 3:
            continue
        if f[0] == "move" and f[1].startswith(f"{side}a:"):
            tera = ""
            start_look = max(0, j - 3)
            end_look = min(len(lines), j + 3)
            if any("terastallize" in lines[k] and side in lines[k] for k in range(start_look, end_look)):
                tera = " terastallize"
            action = f"move {f[2]}{tera}"
            break
        if f[0] == "switch" and f[1].startswith(f"{side}a:"):
            pokemon = f[1].split(": ", 1)[1] if ": " in f[1] else f[1]
            action = f"switch {pokemon}"
            break

    if not action:
        return {"text": ""}

    log_prefix = "\n".join(lines[:turn_line_idx + 1])
    if len(log_prefix) > MAX_LOG_CHARS:
        return {"text": ""}

    messages = [
        {"role": "system", "content": SYSTEM_TEMPLATE.format(side=side)},
        {"role": "user", "content": log_prefix},
        {"role": "assistant", "content": action},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}


print("Streaming replay logs without materializing the full corpus...")
stream = load_dataset(
    "milkkarten/pokemon-showdown-replays-merged",
    split="train",
    streaming=True,
)
stream = stream.shuffle(seed=3407, buffer_size=SHUFFLE_BUFFER)
stream = stream.filter(
    lambda row: "gen9" in str(row.get("formatid") or "").lower().replace(" ", "")
    and (row.get("rating") or 0) >= MIN_RATING
)

train_samples = []
scanned = 0
for row in stream:
    scanned += 1
    formatted = format_sample(row)
    if formatted["text"]:
        train_samples.append(formatted)
    if len(train_samples) >= MAX_TRAIN_SAMPLES:
        break

if not train_samples:
    raise RuntimeError("No training samples were collected. Check dataset access, filters, or disk/network configuration.")

train_dataset = Dataset.from_list(train_samples).shuffle(seed=3407)
print(f"collected_examples={len(train_dataset)}")
print(f"replays_scanned={scanned}")
print(train_dataset[0]["text"][:1000])


## Step 9 — Phase 3a: Small-batch SFT (50 steps)

This is the **notebook demo** — enough to learn the pipeline and beat Random in battle eval. **Full SFT** (millions of samples) runs offline and is published on Hugging Face (Step 13); you can skip this cell and load the Hub adapter instead.


In [ ]:
from pathlib import Path
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported

output_dir = os.environ.get("POKEMON_SFT_OUTPUT", "outputs/pokemon_showdown_agent_tutorial")
if not os.access(output_dir, os.W_OK):
    output_dir = str(WORK_ROOT / "logs" / "sft_tutorial")
Path(output_dir).mkdir(parents=True, exist_ok=True)
print(f"SFT output_dir={output_dir}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=50,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_torch",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=output_dir,
        save_steps=50,
        save_total_limit=2,
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(trainer_stats)


## Step 10 — Inference sanity check

Checks format, log consistency, and optional `|request|` legality. Compare against Step 5 `comparison_rows` baselines when available.


In [ ]:
import torch

model.eval()
FastLanguageModel.for_inference(model)

sys_msg = (
    "You are a Pokemon Showdown battle AI. You play as p2. "
    "Given the battle log, output your next action. "
    "Format: move <name> OR switch <name>. "
    "Append terastallize if you terastallize this turn."
)
user_msg = tutorial_demo_log_with_request()

messages = [
    {"role": "system", "content": sys_msg},
    {"role": "user", "content": user_msg},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )

full_response = tokenizer.decode(outputs[0], skip_special_tokens=False)
if "<|im_start|>assistant\n" in full_response:
    response = full_response.split("<|im_start|>assistant\n")[-1]
else:
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
response = postprocess_agent_response(response)

first_line = response.splitlines()[0].strip() if response.strip() else ""
checks = validate_action_against_log(first_line, user_msg)

print("--- Model prediction ---")
print(response)
print("--- legality & format ---")
print(f"structure_ok={checks['structure_ok']}")
print(f"single_line_ok={checks['single_line_ok']}")
print(f"request_present={checks['request_present']}")
if checks["move_name_nonempty"] is not None:
    print(f"move_name_nonempty={checks['move_name_nonempty']}")
if checks["switch_target_ok"] is not None:
    print(f"switch_target_ok={checks['switch_target_ok']}")
if checks["move_legal_in_request"] is not None:
    print(f"move_legal_in_request={checks['move_legal_in_request']}")
if checks["switch_legal_in_request"] is not None:
    print(f"switch_legal_in_request={checks['switch_legal_in_request']}")
if checks.get("parsed") and checks["parsed"].get("tera") and checks["tera_legal_in_request"] is not None:
    print(f"tera_legal_in_request={checks['tera_legal_in_request']}")
print(f"notes={checks['notes']}")

if comparison_rows:
    print("\nStep 5 baselines (same prompt):")
    for row in comparison_rows:
        name = row.get("label") or row.get("checkpoint", "?")
        axis = row.get("axis", "")
        prefix = f"[{axis}] " if axis else ""
        print(f"  {prefix}{name}: {row['action']}")


## Step 11 — Mini-eval (reference protocol)

Reports valid-format, type-match, and exact-match metrics on a small held-out sample set.


In [ ]:
import random

import torch
from datasets import load_dataset

EVAL_SEED = 3407
EVAL_SAMPLES = 24
random.seed(EVAL_SEED)
torch.manual_seed(EVAL_SEED)

print("Streaming held-out eval examples...")
stream = load_dataset(
    "milkkarten/pokemon-showdown-replays-merged",
    split="train",
    streaming=True,
)
stream = stream.shuffle(seed=EVAL_SEED + 11, buffer_size=10_000)
eval_samples = build_test_samples(stream, EVAL_SAMPLES, seed=EVAL_SEED, max_scans=400_000)
print(f"collected_eval_samples={len(eval_samples)}")

if not eval_samples:
    stream2 = load_dataset(
        "milkkarten/pokemon-showdown-replays-merged",
        split="train",
        streaming=True,
    ).shuffle(seed=EVAL_SEED + 99, buffer_size=10_000)
    eval_samples = build_test_samples(
        stream2, EVAL_SAMPLES, seed=EVAL_SEED, require_gen9=False, max_scans=400_000,
    )
    print(f"collected_eval_samples_relaxed_gen={len(eval_samples)}")

if not eval_samples:
    raise RuntimeError("No eval samples collected; check dataset access or filters.")

model.eval()
metrics, _rows = eval_showdown_agent_batch(
    model, tokenizer, eval_samples, max_new_tokens=30, use_cache=False,
)

print_metrics_summary(metrics, title="Tutorial SFT checkpoint — mini-eval")
out_path = WORK_ROOT / output_dir / "eval_tutorial_protocol.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
save_metrics_json(
    str(out_path),
    metrics,
    extra={
        "protocol": "mini_eval",
        "eval_samples": EVAL_SAMPLES,
        "eval_seed": EVAL_SEED,
        "output_dir": output_dir,
        "trainer_max_steps": 50,
    },
)
print(f"Wrote metrics to {out_path}")


In [ ]:
# Step 12 — GRPO knobs (quick vs full)
import os

_mode = os.environ.get("POKEMON_TUTORIAL_MODE", "quick").strip().lower()
if _mode == "full":
    os.environ.setdefault("GRPO_MAX_STEPS", "300")
    os.environ.setdefault("GRPO_NUM_SAMPLES", "2048")
    os.environ.setdefault("POKEMON_RUN_GRPO", "1")
else:
    os.environ.setdefault("GRPO_MAX_STEPS", "20")
    os.environ.setdefault("GRPO_NUM_SAMPLES", "64")
    os.environ.setdefault("POKEMON_RUN_GRPO", "0")

os.environ.setdefault("GRPO_MAX_PROMPT_TOKENS", "1536")
os.environ.setdefault("GRPO_MAX_COMPLETION", "64")
os.environ.setdefault("GRPO_MIN_RATING", "1200")
print(
    f"POKEMON_TUTORIAL_MODE={_mode}  GRPO_MAX_STEPS={os.environ['GRPO_MAX_STEPS']}  "
    f"GRPO_NUM_SAMPLES={os.environ['GRPO_NUM_SAMPLES']}"
)


<a id="step12"></a>

## Step 12 — GRPO smoke (optional)

Optional GRPO pass on mid-game replay prefixes (same format as SFT). Reward combines format, type hints, and ground-truth action shaping.

> **Quick vs full:** In **`quick`** mode, Step 12b runs the **reward demo** by default (`POKEMON_RUN_GRPO=0`). Set **`POKEMON_RUN_GRPO=1`** to run a short GRPO smoke train (~20 steps). **`full`** mode enables GRPO training by default (300 steps).

> **Re-run note:** Step 13 reloads the model for inference. To run Step 12b GRPO again in the same kernel, restart the kernel or re-run from **Step 7** first.

**Hub assets (optional shortcut):**

| Asset | Hugging Face |
|------|----------------|
| GRPO dataset | [`GoldenGrapeGentleman1/pokemon-showdown-grpo-tutorial`](https://huggingface.co/datasets/GoldenGrapeGentleman1/pokemon-showdown-grpo-tutorial) |
| Tutorial SFT | [`GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-sft`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-sft) |
| Tutorial GRPO | [`GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-grpo`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-grpo) |
| Battle GRPO | [`GoldenGrapeGentleman1/pokemon-showdown-agent-battle-grpo`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-battle-grpo) |

**Modes:**
- `quick`: 64 prompts, ~20 GRPO steps when `POKEMON_RUN_GRPO=1` (demo-only by default)
- `full`: 2048 prompts, 300 steps (`POKEMON_RUN_GRPO=1` by default)

| Sub-step | Content |
|--------|------|
| **12a** | Build GRPO prompts from replays |
| **12b** | Reward demo + short GRPO train |
| **12c** | Before/after mini-eval |

GRPO records load from Hugging Face by default (`POKEMON_USE_HF_ASSETS=1`). Step 12a can also build records from the replay stream or read local `data/grpo_tutorial_*.jsonl` when `POKEMON_USE_HF_ASSETS=0`.

Extended **battle-outcome GRPO** (Champion tier) is trained offline with full SFT + real battle rewards — use the Hub [`pokemon-showdown-agent-battle-grpo`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-battle-grpo) checkpoint in Steps 14–15, or set `POKEMON_GRPO_CKPT` for a local adapter.


In [ ]:
import json
import os
import sys
from pathlib import Path

from IPython.display import HTML, display

from datasets import load_dataset
TUTORIAL_MODE = os.environ.get("POKEMON_TUTORIAL_MODE", "quick").strip().lower()
USE_HF = os.environ.get("POKEMON_USE_HF_ASSETS", "1").strip() == "1"
HF_DATASET = os.environ.get("POKEMON_HF_GRPO_DATASET", HF_GRPO_DATASET_DEFAULT)
LOCAL_DEMO = WORK_ROOT / "data/grpo_tutorial_demo.jsonl"
LOCAL_TRAIN = WORK_ROOT / "data/grpo_tutorial_train.jsonl"
MAX_PROMPT_TOKENS = int(os.environ.get("GRPO_MAX_PROMPT_TOKENS", "1536"))
MIN_RATING = int(os.environ.get("GRPO_MIN_RATING", "1200"))
TARGET_N = int(os.environ.get("GRPO_NUM_SAMPLES", "64"))

# --- 12a.1 Replay -> mid-game prefix + gt_action ---
print("=== GRPO data pipeline (SFT replay format) ===")
try:
    _stream = load_dataset(
        "milkkarten/pokemon-showdown-replays-merged", split="train", streaming=True
    ).shuffle(seed=3407, buffer_size=5000)
    _demo_recs = collect_grpo_sft_records_from_stream(
        _stream, 1, min_rating=MIN_RATING, max_scans=50_000
    )
    if _demo_recs:
        _r = _demo_recs[0]
        print(f"side={_r['side']} rating={_r['rating']} gt={_r.get('gt_action')!r}")
        print(f"log_chars={len(_r['log'])} schema={_r.get('schema', GRPO_SCHEMA_SFT)}")
        print(_r["log"][-350:])
except Exception as _exc:
    print(f"(stream demo skipped: {_exc})")

# --- 12a.2 Load dataset: local jsonl (if present) > Hub > stream fallback ---
_split = "demo" if TUTORIAL_MODE != "full" else "train"
_local = LOCAL_DEMO if _split == "demo" else LOCAL_TRAIN
_grpo_records = []
try:
    _grpo_records = load_grpo_tutorial_records(
        hf_dataset=HF_DATASET, split=_split, local_path=str(_local), use_hf=USE_HF
    )
    print(f"\nloaded {len(_grpo_records)} records from {'HF '+HF_DATASET+'/'+_split if USE_HF else _local}")
except Exception as exc:
    print(f"\nHub/local failed ({exc}); building {TARGET_N} from stream...")
    _stream = load_dataset(
        "milkkarten/pokemon-showdown-replays-merged", split="train", streaming=True
    ).shuffle(seed=3407, buffer_size=10_000)
    _grpo_records = collect_grpo_sft_records_from_stream(
        _stream, TARGET_N, min_rating=MIN_RATING, max_scans=200_000
    )

_grpo_dataset_rows = grpo_rows_for_trainer(
    _grpo_records, tokenizer, max_prompt_tokens=MAX_PROMPT_TOKENS
)[:TARGET_N]
_grpo_dataset_prompts = _grpo_dataset_rows  # alias for Step 12b

if len(_grpo_dataset_rows) < 4:
    raise RuntimeError(
        f"Too few GRPO rows ({len(_grpo_dataset_rows)}). "
        f"Set POKEMON_USE_HF_ASSETS=1 or lower GRPO_MIN_RATING"
    )

_unique = len({str(r["prompt"])[:200] for r in _grpo_dataset_rows})
_html = "<table border='1' cellpadding='4' style='border-collapse:collapse;font-size:12px'>"
_html += "<tr><th>side</th><th>gt_action</th><th>tokens</th><th>log tail</th></tr>"
for row in _grpo_dataset_rows[:8]:
    _rendered = tokenizer.apply_chat_template(
        row["prompt"], tokenize=False, add_generation_prompt=True, enable_thinking=False
    ) if isinstance(row["prompt"], list) else row["prompt"]
    _nt = len(tokenizer(_rendered, add_special_tokens=False)["input_ids"])
    _log = row["prompt"][1]["content"][-80:].replace("\n", " ") if isinstance(row["prompt"], list) else _rendered[-80:]
    _html += f"<tr><td>{row['side']}</td><td><code>{row.get('gt_action','')[:30]}</code></td><td>{_nt}</td><td><code>{_log}</code></td></tr>"
_html += "</table>"
display(HTML(_html))
print(
    f"Step 12a OK — rows={len(_grpo_dataset_rows)} unique={_unique} "
    f"mode={TUTORIAL_MODE} min_rating={MIN_RATING}"
)


In [ ]:
import json
import os
import gc
import sys
from pathlib import Path

import torch
from datasets import Dataset
from IPython.display import HTML, display
from unsloth import is_bfloat16_supported

TYPE_CHART = load_type_chart()
showdown_reward_func = make_showdown_grpo_reward(TYPE_CHART)

# --- Reward demo (format + type + gt) ---
_demo_log = (
    "|turn|3\n|switch|p1a: Garchomp|Garchomp, M|100/100\n"
    "|switch|p2a: Corviknight|Corviknight, M|100/100\n"
)
_demo_prompt = (
    "<|im_start|>system\nYou are a Pokemon Showdown battle AI. You play as p2.\n"
    f"<|im_start|>user\n{_demo_log}\n<|im_start|>assistant\n"
)
_gt = "move Brave Bird"
_demo_rows = [
    ("move Brave Bird", "format + exact gt"),
    ("move Earthquake", "format OK, gt differs"),
    ("switch Dragapult", "⚠️ switch vs gt move"),
    ("hello world", "invalid format"),
]
_html = "<table border='1' cellpadding='6' style='border-collapse:collapse;font-size:13px'>"
_html += "<tr><th>Output</th><th>Note</th><th>Reward</th></tr>"
for action, note in _demo_rows:
    rw = showdown_reward_func(
        [_demo_prompt], [[{"content": action}]], gt_action=[_gt], side=["p2"]
    )[0]
    bg = "#dfd" if rw >= 5 else ("#ffd" if rw >= 0 else "#fdd")
    _html += f"<tr style='background:{bg}'><td><code>{action}</code></td><td>{note}</td><td><b>{rw}</b></td></tr>"
_html += "</table>"
display(HTML(_html))

grpo_trainer = None
_grpo_before = None
_grpo_after = None
RUN_GRPO = os.environ.get("POKEMON_RUN_GRPO", "0").strip() == "1"
GRPO_MAX_STEPS = int(os.environ.get("GRPO_MAX_STEPS", "20"))
GRPO_NUM_SAMPLES = int(os.environ.get("GRPO_NUM_SAMPLES", "64"))
GRPO_MAX_COMPLETION = int(os.environ.get("GRPO_MAX_COMPLETION", "64"))
GRPO_MAX_PROMPT_TOKENS = int(os.environ.get("GRPO_MAX_PROMPT_TOKENS", "1536"))


def _grpo_eval_generate(row: dict) -> dict:
    """Generate on one GRPO row (uses gt_action for reward only, not in prompt)."""
    prompt = row["prompt"]
    if isinstance(prompt, list):
        text = tokenizer.apply_chat_template(
            prompt, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    else:
        text = prompt
    inp = tokenizer(text, return_tensors="pt").to("cuda")
    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=48,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False,
        )
    resp = postprocess_agent_response(
        tokenizer.decode(out[0][inp.input_ids.shape[-1]:], skip_special_tokens=True)
    )
    cmd, _ = extract_cmd(resp)
    line = cmd or (resp.splitlines()[0].strip() if resp.strip() else "")
    reward = showdown_reward_func(
        [text],
        [[{"content": resp}]],
        gt_action=[row.get("gt_action", "")],
        side=[row.get("side", "p2")],
    )[0]
    return {"line": line, "reward": reward, "response": resp, "gt": row.get("gt_action", "")}


_eval_row = _grpo_dataset_rows[0] if _grpo_dataset_rows else None
if _eval_row is None:
    raise RuntimeError("Run Step 12a first.")

print("[12b] pre-GRPO eval on held-out GRPO row...", flush=True)
_grpo_before = _grpo_eval_generate(_eval_row)
print(
    f"Before: pred={_grpo_before['line']!r} gt={_grpo_before['gt']!r} reward={_grpo_before['reward']}",
    flush=True,
)

if not RUN_GRPO:
    print("GRPO skipped. Set POKEMON_RUN_GRPO=1 and re-run this cell.")
else:
    from trl import GRPOConfig, GRPOTrainer

    def _patch_live_grpo_module() -> None:
        import sys
        import torch

        try:
            from unsloth_zoo.device_type import device_synchronize
        except ImportError:
            def device_synchronize():
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                elif hasattr(torch, "xpu") and torch.xpu.is_available():
                    torch.xpu.synchronize()

        mods = [m for k, m in sys.modules.items() if k.endswith("UnslothGRPOTrainer")]
        if not mods:
            cache_dir = str(WORK_ROOT / "unsloth_compiled_cache")
            if cache_dir not in sys.path:
                sys.path.insert(0, cache_dir)
            try:
                import UnslothGRPOTrainer as _m
                mods = [_m]
            except ModuleNotFoundError:
                return  # compiled on first GRPOTrainer(); re-run patch after trainer init if needed
        for mod in mods:
            if not hasattr(mod, "align_completion_tool_mask"):
                def align_completion_tool_mask(tool_mask, completion_mask, _mod=mod):
                    if tool_mask is None:
                        return completion_mask
                    tool_mask = tool_mask.to(device=completion_mask.device)
                    if tool_mask.shape == completion_mask.shape:
                        aligned = tool_mask
                    else:
                        aligned = _mod.align_logprobs_with_mask(
                            tool_mask, completion_mask, pad_value=0
                        )
                    return completion_mask * aligned.to(dtype=completion_mask.dtype)

                mod.align_completion_tool_mask = align_completion_tool_mask
            mod.device_synchronize = device_synchronize

    _patch_live_grpo_module()

    if not _grpo_dataset_rows:
        raise RuntimeError("Run Step 12a first.")
    _train_rows = _grpo_dataset_rows[:GRPO_NUM_SAMPLES]
    print(f"GRPO rows={len(_train_rows)} max_steps={GRPO_MAX_STEPS}", flush=True)

    grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[showdown_reward_func],
        args=GRPOConfig(
            output_dir=str(WORK_ROOT / "grpo_outputs"),
            learning_rate=5e-6,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            num_generations=4,
            max_prompt_length=GRPO_MAX_PROMPT_TOKENS,
            max_completion_length=GRPO_MAX_COMPLETION,
            temperature=0.7,
            beta=0.04,
            max_steps=GRPO_MAX_STEPS,
            logging_steps=1,
            report_to="none",
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
        ),
        train_dataset=Dataset.from_list(_train_rows),
    )
    _patch_live_grpo_module()
    print("Starting GRPO train...", flush=True)
    grpo_stats = grpo_trainer.train()
    print(grpo_stats, flush=True)

    model = grpo_trainer.model
    model.eval()
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    gc.collect()

    print("[12b] post-GRPO eval...", flush=True)
    _grpo_after = _grpo_eval_generate(_eval_row)
    print(
        f"After: pred={_grpo_after['line']!r} gt={_grpo_after['gt']!r} reward={_grpo_after['reward']}",
        flush=True,
    )


In [ ]:
import os
import gc
import torch

from datasets import load_dataset

print("[12c] post-GRPO mini-eval...", flush=True)

if _grpo_before is None:
    print("Step 12c skipped — run Step 12b first.", flush=True)
else:
    if _grpo_after is not None:
        print(f"  before reward={_grpo_before['reward']}  after reward={_grpo_after['reward']}")
        print(f"  before pred={_grpo_before['line']!r}  after pred={_grpo_after['line']!r}")
        print(f"  gt={_grpo_before['gt']!r}")
        demo_log = tutorial_demo_log_with_request()
        _pre_checks = validate_action_against_log(_grpo_before["line"], demo_log)
        _post_checks = validate_action_against_log(_grpo_after["line"], demo_log)
        print(f"  before move_legal={_pre_checks.get('move_legal_in_request')}")
        print(f"  after  move_legal={_post_checks.get('move_legal_in_request')}")

    RUN_12C = os.environ.get("POKEMON_RUN_12C_EVAL", "1").strip() == "1"
    if not RUN_12C:
        print("[12c] extended mini-eval skipped (set POKEMON_RUN_12C_EVAL=1).", flush=True)
    elif grpo_trainer is None and _grpo_after is None:
        print("[12c] GRPO train skipped (POKEMON_RUN_GRPO=0) — demo-only pre-eval above.", flush=True)
    elif grpo_trainer is not None:
        _eval_samples = []
        for row in _grpo_dataset_rows[: min(12, len(_grpo_dataset_rows))]:
            if not row.get("gt_action"):
                continue
            if isinstance(row["prompt"], list):
                _log = next(m["content"] for m in row["prompt"] if m.get("role") == "user")
            else:
                _log = row["prompt"].split("<|im_start|>user\n", 1)[-1].split("<|im_start|>", 1)[0].strip()
            _eval_samples.append({
                "side": row["side"],
                "prompt": _log,
                "gt_action": row["gt_action"],
                "rating": 1500,
                "format": "gen9",
            })
        if _eval_samples:
            model = grpo_trainer.model
            model.eval()
            torch.cuda.empty_cache()
            gc.collect()
            _metrics, _ = eval_showdown_agent_batch(
                model, tokenizer, _eval_samples, max_new_tokens=30, use_cache=False
            )
            print_metrics_summary(_metrics, title=f"Post-GRPO mini-eval ({len(_eval_samples)} samples)")
        _quick_n = 4
        print(f"\n[12c] streaming dataset ({_quick_n} samples)...", flush=True)
        _stream = load_dataset("milkkarten/pokemon-showdown-replays-merged", split="train", streaming=True)
        _stream = _stream.shuffle(seed=3407 + 99, buffer_size=5000)
        _qs = build_test_samples(_stream, _quick_n, seed=3407 + 99, max_scans=20_000)
        if _qs:
            model = grpo_trainer.model
            model.eval()
            _m, _ = eval_showdown_agent_batch(model, tokenizer, _qs, max_new_tokens=30, use_cache=False)
            print_metrics_summary(_m, title=f"Post-GRPO quick eval ({len(_qs)} samples)")
        else:
            print("[12c] no streaming eval samples collected.")
    print("Step 12c OK", flush=True)


In [ ]:
import os

# Kept for notebook cell-id compatibility; Step 12c logic lives in the previous cell.
if _grpo_before is None:
    print("Step 12c skipped — run Step 12b first.", flush=True)
else:
    print("Step 12c complete.", flush=True)


## Step 13 — Export & publish to Hugging Face

Optional Hub upload and tier-aware model loading before battle eval.

| Action | When |
|--------|------|
| Local jsonl for GRPO | Step **12a** builds or loads `data/grpo_tutorial_*.jsonl`; Hub dataset is the default shortcut |
| Upload checkpoint to Hub | Optional — use `hf upload` (see next cell) when `HF_TOKEN` is set |

Run the tier load cell below before Step 14 battle eval.


In [ ]:
# Optional — upload your local checkpoint to Hugging Face (requires HF_TOKEN)
# hf auth login
# hf upload your-user/pokemon-showdown-agent-tutorial-sft outputs/pokemon_showdown_agent_tutorial/checkpoint-50 --repo-type model
print('Skip unless you want to publish your tutorial checkpoint.')


In [ ]:
# Load adapter for battle eval (tier-aware)
from pathlib import Path
from unsloth import FastLanguageModel

_tier = os.environ.get("POKEMON_AGENT_TIER", "enthusiast")
_hf_sft = os.environ.get("POKEMON_HF_SFT_MODEL", "GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-sft")
_hf_full = os.environ.get("POKEMON_HF_FULL_SFT", "GoldenGrapeGentleman1/pokemon-showdown-agent-full-sft")
_hf_battle = os.environ.get("POKEMON_HF_BATTLE_GRPO", "GoldenGrapeGentleman1/pokemon-showdown-agent-battle-grpo")

def _discover_battle_grpo_ckpt() -> Path | None:
    env = os.environ.get("POKEMON_GRPO_CKPT", "").strip()
    if env and Path(env).is_dir():
        return Path(env)
    for parent in sorted(WORK_ROOT.glob("grpo_*_outputs")):
        for ckpt in sorted(parent.glob("checkpoint-*"), reverse=True):
            if (ckpt / "adapter_config.json").is_file():
                return ckpt
    return None

def _local_sft_ckpt() -> Path | None:
    for p in [
        WORK_ROOT / "outputs/pokemon_showdown_agent_tutorial/checkpoint-50",
        WORK_ROOT / "logs/sft_tutorial/checkpoint-50",
    ]:
        if p.is_dir() and (p / "adapter_config.json").is_file():
            return p
    return None

_local_sft = _local_sft_ckpt()
_battle_grpo = _discover_battle_grpo_ckpt()
print(f"Tier={_tier}  loading Qwen3-4B base ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-4B",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,
    attn_implementation="sdpa",
)
if _tier == "champion":
    if _battle_grpo is not None:
        print(f"Champion: local battle GRPO {_battle_grpo}")
        model.load_adapter(str(_battle_grpo), adapter_name="grpo")
    else:
        print(f"Champion: Hub battle GRPO {_hf_battle}")
        model.load_adapter(_hf_battle, adapter_name="grpo")
    model.set_adapter("grpo")
elif _tier == "competitive":
    print(f"Competitive: full SFT {_hf_full}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=_hf_full,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=False,
        attn_implementation="sdpa",
    )
else:
    _tier_label = "Casual" if _tier == "casual" else "Enthusiast"
    if _local_sft is not None:
        print(f"{_tier_label}: local SFT {_local_sft}")
        model.load_adapter(str(_local_sft), adapter_name="sft")
    else:
        print(f"{_tier_label}: Hub SFT {_hf_sft}")
        model.load_adapter(_hf_sft, adapter_name="sft")
    model.set_adapter("sft")
print("Ready for Step 14 battle eval or Step 15 friendly play")


### Step 13b — Battle eval helpers (inline)

Defines `LLMPlayer` (model picks moves via poke-env), checkpoint resolution, and async win-rate eval against Random / SimpleHeuristics / MaxBasePower baselines.


In [ ]:
# Battle win-rate eval (LLM vs poke-env baselines)
from __future__ import annotations

from collections import defaultdict
import argparse
import asyncio
import json
import os
import random
import sys
import time
from pathlib import Path
from typing import Any

import torch

try:
    from poke_env.battle.abstract_battle import AbstractBattle
    from poke_env.battle.battle import Battle
except ModuleNotFoundError:  # poke-env < 0.9
    from poke_env.environment.abstract_battle import AbstractBattle
    from poke_env.environment.battle import Battle

from poke_env.ps_client.account_configuration import AccountConfiguration
from poke_env.concurrency import POKE_LOOP
from poke_env.player import MaxBasePowerPlayer, Player, RandomPlayer, SimpleHeuristicsPlayer
from poke_env.player.battle_order import BattleOrder
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template

WORK_ROOT = Path(os.environ.get("POKEMON_WORK_ROOT", Path.cwd()))

HF_SFT_DEFAULT = os.environ.get(
    "POKEMON_HF_SFT_MODEL", "GoldenGrapeGentleman1/pokemon-showdown-agent-tutorial-sft"
)
HF_BATTLE_GRPO_DEFAULT = os.environ.get(
    "POKEMON_HF_BATTLE_GRPO", "GoldenGrapeGentleman1/pokemon-showdown-agent-battle-grpo"
)


def resolve_tutorial_sft_ckpt(work_root: Path | None = None) -> str:
    """Local tutorial SFT checkpoint, else Hub tutorial-sft adapter."""
    work = work_root or WORK_ROOT
    env = os.environ.get("POKEMON_SFT_CKPT", "").strip()
    if env:
        return env
    for cand in (
        work / "outputs/pokemon_showdown_agent_tutorial/checkpoint-50",
        work / "logs/sft_tutorial/checkpoint-50",
    ):
        if (cand / "adapter_config.json").is_file():
            return str(cand)
    return HF_SFT_DEFAULT


def resolve_battle_grpo_ckpt(work_root: Path | None = None) -> str:
    """Local GRPO adapter if present, else Hub battle-grpo."""
    work = work_root or WORK_ROOT
    env = os.environ.get("POKEMON_GRPO_CKPT", "").strip()
    if env:
        return env
    for parent in sorted(work.glob("grpo_*_outputs")):
        for ckpt in sorted(parent.glob("checkpoint-*"), reverse=True):
            if (ckpt / "adapter_config.json").is_file():
                return str(ckpt)
    return HF_BATTLE_GRPO_DEFAULT


SFT_CKPT = resolve_tutorial_sft_ckpt()
GRPO_CKPT = resolve_battle_grpo_ckpt()
BATTLE_FORMAT = os.environ.get("POKE_BATTLE_FORMAT", "gen9randombattle")
SERVER = (
    os.environ.get("POKE_SHOWDOWN_SERVER")
    or os.environ.get("POKEMON_SHOWDOWN_SERVER", "local")
).strip().lower()
GRPO_COLLECT_EXPERT = os.environ.get("GRPO_COLLECT_EXPERT", "0").strip() == "1"

OPPONENTS: dict[str, type] = {
    "Random": RandomPlayer,
    "SimpleHeuristics": SimpleHeuristicsPlayer,
    "MaxBasePower": MaxBasePowerPlayer,
}


def resolve_adapter_ref(ref: str | Path) -> str | None:
    p = Path(ref)
    if p.is_dir() and (p / "adapter_config.json").is_file():
        return str(p)
    text = str(ref).strip()
    if "/" in text and not p.exists():
        return text
    return None


def battle_log_text(battle: Battle) -> str:
    replay = getattr(battle, "_replay_data", None) or getattr(battle, "replay_data", None)
    if replay:
        if isinstance(replay, list) and replay and isinstance(replay[0], list):
            lines = ["|" + "|".join(str(x) for x in msg if x) for msg in replay]
            return truncate_log_tail("\n".join(lines))
        return truncate_log_tail(str(replay))
    lines: list[str] = []
    obs = getattr(battle, "observations", {}) or {}
    for turn in sorted(obs.keys()):
        entry = obs[turn]
        if isinstance(entry, (list, tuple)):
            for event in entry:
                lines.append(str(event))
            continue
        events = getattr(entry, "events", None)
        if events:
            for event in events:
                if isinstance(event, (list, tuple)):
                    parts = [str(x) for x in event if x is not None and str(x) != ""]
                    if parts:
                        lines.append("|" + "|".join(parts))
                else:
                    lines.append(str(event))
        else:
            lines.append(str(entry))
    if lines:
        return truncate_log_tail("\n".join(lines))
    return f"|turn|{getattr(battle, 'turn', 0) or 0}"


def _can_tera(battle: Battle) -> bool:
    return bool(
        getattr(battle, "can_tera", False) or getattr(battle, "can_terastallize", False)
    )


def legal_action_strings(battle: Battle) -> list[str]:
    legal: list[str] = []
    if battle.available_moves:
        for move in battle.available_moves:
            base = f"move {move.id}"
            legal.append(base)
            if _can_tera(battle):
                legal.append(f"{base} terastallize")
    if battle.available_switches:
        for mon in battle.available_switches:
            legal.append(f"switch {mon.species}")
    return legal


def order_from_action(battle: Battle, action: str) -> BattleOrder | None:
    action = action.strip()
    low = action.lower()
    if low.startswith("move "):
        name = action[5:].replace(" terastallize", "").replace(" Terastallize", "").strip()
        tera = "terastallize" in low
        for move in battle.available_moves or []:
            mid = str(getattr(move, "id", move)).lower()
            if mid == name.lower() or name.lower() in mid:
                return Player.create_order(move, terastallize=tera)
    if low.startswith("switch "):
        name = action[7:].strip()
        for mon in battle.available_switches or []:
            if str(mon.species).lower() == name.lower():
                return Player.create_order(mon)
    return None


class LLMPlayer(Player):
    def __init__(self, model, tokenizer, *, side_hint: str = "p2", verbose: bool = False, **kwargs):
        super().__init__(**kwargs)
        self._model = model
        self._tokenizer = tokenizer
        self._side_hint = side_hint
        self._device = next(model.parameters()).device
        self._verbose = verbose
        self._turn_counts: dict[str, int] = defaultdict(int)

    def choose_move(self, battle: AbstractBattle) -> BattleOrder:
        if not isinstance(battle, Battle):
            return self.choose_random_move(battle)
        if battle.in_team_preview or (battle.force_switch and battle.available_switches):
            return self.choose_random_move(battle)

        side = battle.player_role or self._side_hint
        log = battle_log_text(battle)
        legal = legal_action_strings(battle)
        if not legal:
            return self.choose_random_move(battle)

        legal_block = "\n".join(f"- {a}" for a in legal)
        user = (
            f"{log}\n\nLegal actions this turn (pick exactly one):\n{legal_block}\n"
            "Reply with one line only."
        )
        system = SYSTEM_TEMPLATE.format(side=side) + " Pick one action from the legal list."
        prompt = self._tokenizer.apply_chat_template(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = self._tokenizer(prompt, return_tensors="pt").to(self._device)
        with torch.no_grad():
            out = self._model.generate(
                **inputs,
                max_new_tokens=48,
                do_sample=False,
                use_cache=True,
            )
        raw = self._tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
        text = postprocess_agent_response(raw)
        line = text.strip().splitlines()[0] if text.strip() else ""
        cmd, _ = extract_cmd(line)
        order = order_from_action(battle, line) if line else None
        if order is None and cmd:
            for action in legal:
                if cmd.lower() in action.lower():
                    order = order_from_action(battle, action)
                    if order is not None:
                        break
        if order is None and legal:
            order = order_from_action(battle, legal[0])
        if order is not None:
            return order
        return self.choose_random_move(battle)


def load_llm_base():
    print("Loading Qwen3-4B weights ...", flush=True)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="Qwen/Qwen3-4B",
        max_seq_length=2048,
        dtype=torch.bfloat16 if is_bfloat16_supported() else torch.float16,
        load_in_4bit=False,
        attn_implementation="sdpa",
    )
    tokenizer = get_chat_template(tokenizer, chat_template="qwen3")
    model = FastLanguageModel.get_peft_model(
        model,
        r=64,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=128,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    return model, tokenizer


def make_player_kwargs(name_key: str) -> dict[str, Any]:
    kw = player_kwargs(name_key, server=SERVER)
    kw["battle_format"] = BATTLE_FORMAT
    return kw


async def close_player(player: Player) -> None:
    try:
        await player.ps_client.stop_listening()
    except Exception:
        pass
    try:
        pending = [
            t for t in asyncio.all_tasks(POKE_LOOP)
            if not t.done() and t is not asyncio.current_task(POKE_LOOP)
        ]
        handle = []
        for task in pending:
            coro = task.get_coro()
            qual = getattr(coro, "__qualname__", "") if coro is not None else ""
            if "_handle_message" in qual:
                task.cancel()
                handle.append(task)
        if handle:
            await asyncio.gather(*handle, return_exceptions=True)
    except Exception:
        pass
    await asyncio.sleep(0.05)


async def run_matchups(
    llm_label: str,
    model,
    tokenizer,
    opponent_cls,
    opponent_label: str,
    n_battles: int,
    *,
    progress: bool = True,
) -> dict[str, Any]:
    llm = LLMPlayer(model, tokenizer, **make_player_kwargs(f"L{llm_label}"))
    opp = opponent_cls(**make_player_kwargs(f"O{opponent_label[:4]}"))
    t0 = time.time()
    wins = 0
    try:
        for i in range(1, n_battles + 1):
            w_before = sum(1 for b in llm.battles.values() if b.won)
            await asyncio.wait_for(llm.battle_against(opp, n_battles=1), timeout=180.0)
            wins = sum(1 for b in llm.battles.values() if b.won)
            if progress and n_battles > 1:
                won_this = wins > w_before
                print(
                    f"      [{i}/{n_battles}] {'win' if won_this else 'loss'}"
                    f" — running {wins}/{i} ({100 * wins / i:.0f}%)",
                    flush=True,
                )
    finally:
        llm.reset_battles()
        opp.reset_battles()
        await close_player(llm)
        await close_player(opp)
    elapsed = time.time() - t0
    return {
        "llm": llm_label,
        "opponent": opponent_label,
        "battles": n_battles,
        "wins": wins,
        "losses": n_battles - wins,
        "win_rate": wins / max(n_battles, 1),
        "seconds": round(elapsed, 1),
    }


def print_table(rows: list[dict[str, Any]]) -> None:
    print("\n" + "=" * 72)
    print(f"{'Model':<8} {'Opponent':<18} {'W':>4} {'L':>4} {'Win%':>8} {'Time(s)':>10}")
    print("-" * 72)
    for r in rows:
        print(
            f"{r['llm']:<8} {r['opponent']:<18} {r['wins']:>4} {r['losses']:>4} "
            f"{100*r['win_rate']:>7.1f}% {r['seconds']:>10.1f}"
        )
    print("=" * 72)


def parse_opponents(spec: str) -> list[tuple[type, str]]:
    out: list[tuple[type, str]] = []
    for name in spec.split(","):
        name = name.strip()
        if not name:
            continue
        if name not in OPPONENTS:
            raise SystemExit(f"Unknown opponent {name!r}; choose from {list(OPPONENTS)}")
        out.append((OPPONENTS[name], name))
    return out


def run_async(coro):
    fut = asyncio.run_coroutine_threadsafe(coro, POKE_LOOP)
    return fut.result()

async def run_battle_eval(*, sft_ckpt, grpo_ckpt, n_battles, opponents=("Random","SimpleHeuristics","MaxBasePower"), grpo_only=False, sft_only=False, output_path=None):
    patch_guest_authentication(); ensure_local_showdown()
    rows = []
    ckpts = []
    if not grpo_only: ckpts.append(("SFT", sft_ckpt))
    if not sft_only: ckpts.append(("GRPO", grpo_ckpt))
    ckpts = [(l, s) for l, s in ckpts if resolve_adapter_ref(s)]
    if not ckpts:
        raise RuntimeError("No adapters found (local checkpoint or Hub id)")
    opp_map = {k: OPPONENTS[k] for k in opponents}
    for label, ckpt in ckpts:
        print(f"\nAdapter {label} <- {ckpt}")
        model, tok = load_llm_base()
        model.load_adapter(resolve_adapter_ref(ckpt), adapter_name=label)
        model.set_adapter(label); model.eval()
        for opp_name, opp_cls in opp_map.items():
            print(f"  {label} vs {opp_name} x{n_battles} ...", flush=True)
            row = await run_matchups(label, model, tok, opp_cls, opp_name, n_battles)
            rows.append(row)
            print(f"    -> {row['wins']}/{row['battles']} ({100*row['win_rate']:.1f}%)", flush=True)
        del model; torch.cuda.empty_cache()
    print_table(rows)
    if output_path:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        Path(output_path).write_text(json.dumps(rows, indent=2), encoding="utf-8")
        print(f"Wrote {output_path}")
    return rows

async def run_friendly_agent(*, grpo_ckpt, friendly_name="LLMSFTAgent", n_battles=999):
    patch_guest_authentication(); ensure_local_showdown()
    src = resolve_adapter_ref(grpo_ckpt)
    if not src:
        raise RuntimeError("No adapter for friendly mode")
    model, tok = load_llm_base()
    model.load_adapter(src, adapter_name="friendly"); model.set_adapter("friendly"); model.eval()
    kw = make_player_kwargs("F")
    kw["account_configuration"] = resolve_friendly_account(friendly_name)
    player = LLMPlayer(model, tok, **kw)
    print(f"\nFriendly agent online: {player.username}")
    print_showdown_browser_help()
    print(f"  Challenge '{player.username}' to {BATTLE_FORMAT} (interrupt kernel to stop)")
    await player.accept_challenges(None, n_battles)

print("Battle eval helpers ready")


<a id="step14"></a>

## Step 14 — Real battle win-rate eval (poke-env)

Steps 11–12 score **replay protocol** alignment. Here you run full **gen9randombattle** matches — the metric that reflects real ladder strength.

**Before you run:**
1. This cell starts local Showdown automatically (`bootstrap_local_showdown`). Forward port **8000** in Docker / remote IDE if needed.
2. Set `POKEMON_AGENT_TIER` in Step 0 (`casual` / `enthusiast` → tutorial SFT; `competitive` / `champion` → stronger lineups).

**Opponents:** Random (format sanity), **SimpleHeuristics** (type-aware baseline), MaxBasePower (damage baseline).

| Tier | What gets evaluated |
|------|---------------------|
| casual / enthusiast | Tutorial SFT + Hub **battle GRPO** (side-by-side) |
| competitive / champion | Side-by-side **lineup** when local checkpoints exist; otherwise Hub battle GRPO (Champion) |

> **GRPO checkpoint:** enthusiast tier uses Hub [`pokemon-showdown-agent-battle-grpo`](https://huggingface.co/GoldenGrapeGentleman1/pokemon-showdown-agent-battle-grpo) by default. Override with `POKEMON_GRPO_CKPT` for a local adapter.

**Reading results:** Win rate vs Random should be high once training converged. SimpleHeuristics is the harder bar — improvement there means the agent is learning tactics, not just syntax.


In [ ]:
import json
import os
from pathlib import Path

quiet_eval_logs()
print("Showdown:", bootstrap_local_showdown())

_n = int(os.environ.get("POKEMON_N_BATTLES", "10" if os.environ.get("POKEMON_TUTORIAL_MODE") == "quick" else "50"))
_tier = os.environ.get("POKEMON_AGENT_TIER", "enthusiast")
_eval_dir = WORK_ROOT / "logs" / "eval"
_eval_dir.mkdir(parents=True, exist_ok=True)
_out = _eval_dir / f"eval_step14_{_tier}_{_n}.json"

_sft = resolve_tutorial_sft_ckpt()
_grpo = resolve_battle_grpo_ckpt()
_grpo_only = _tier == "champion"

print(f"Running battle eval: tier={_tier} battles={_n}")
print(f"  SFT={_sft}")
print(f"  GRPO={_grpo}")

rows = run_async(run_battle_eval(
    sft_ckpt=_sft,
    grpo_ckpt=_grpo,
    n_battles=_n,
    grpo_only=_grpo_only,
    output_path=str(_out),
))

print(f"\n=== Results ({_out.name}) ===")
for r in rows:
    print(f"  {r['llm']} vs {r['opponent']}: {100*r['win_rate']:.1f}%")


## Step 15 — Play online (friendly + ladder)

### Start local Showdown
Run the **next code cell** once. It installs or clones Showdown if needed, starts the server, and prints the browser URL.

| Environment | What to do |
|-------------|------------|
| Docker | Publish port `-p 8000:8000` when creating the container |
| Remote IDE | **Ports → Forward 8000**, then open the forwarded URL |

This is a **private local server** — not the official ladder at play.pokemonshowdown.com.

### Local friendly (recommended first)
1. Run the Showdown start cell below.
2. Run the friendly-agent cell — it loads the **battle GRPO** adapter (same as Step 14 enthusiast GRPO) and prints **`Friendly agent online: …`** (default base name `LLMSFTAgent` + unique suffix). Challenge **that exact username**.
3. Open the printed testclient URL and challenge the printed username to **gen9randombattle**.

> Re-run Step 15 without interrupting the prior cell? A fixed name hits `|nametaken|` — the friendly agent cell auto-uniqueifies unless `POKEMON_FRIENDLY_UNIQUE=0`.

### Official ladder (Competitive / Champion tier)
1. Create a [Pokemon Showdown](https://play.pokemonshowdown.com/) account (guest accounts cannot ladder reliably).
2. Point the friendly agent at the official server: `POKE_SHOWDOWN_SERVER=official`.
3. **Champion tier:** set `POKEMON_HF_BATTLE_GRPO` (defaults to Hub battle GRPO) or `POKEMON_GRPO_CKPT` for a local adapter.
4. Climb **gen9randombattle** — expect many games; top ranks need sustained win rate over strong opponents.

**Tip:** Keep `enable_thinking=False` during inference so the model emits a single `move` / `switch` line.


In [ ]:
import sys
print(bootstrap_local_showdown())


In [ ]:
# Friendly agent — interrupt kernel to stop
import os

run_async(run_friendly_agent(
    grpo_ckpt=resolve_battle_grpo_ckpt(),
    friendly_name=os.environ.get("POKEMON_FRIENDLY_NAME", "LLMSFTAgent"),
    n_battles=999,
))
